In [158]:
# NOTE that this needs to be started from a jupyter notebook that's within the IRAF27 environment.
import os.path
import os
import subprocess
import shutil
import sys
import glob
from cStringIO import StringIO
import itertools
import collections
from datetime import datetime

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy.modeling import fitting, models
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time
from pyraf import iraf
iraf.set(stdimage="imt2048")
import matplotlib
%matplotlib
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

Using matplotlib backend: Qt5Agg


In [2]:
BINARY_PATH = os.path.expanduser("~/SCIENCE/Binaries")
IMAGE_PATH = os.path.join(BINARY_PATH, "Modspec")
# Iraf tasks can have a maximum of 63 characters. So I don't want to work with absolute paths, just in case.
iraf.cd(IMAGE_PATH)

In [87]:
# Load in the packages we want
iraf.noao()
iraf.imred()
iraf.ccdred()
iraf.twodspec()
iraf.longslit()
iraf.apextract()
iraf.rv()
iraf.longslit.disp = 2
obsnights = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

In [13]:

RAW_FOLDER = "Modspec_Raw"
TRIMMED_FOLDER = "Modspec_Trimmed"
# First copy the raw data before performing operations on it.
#iraf.cp(RAW_FOLDER, TRIMMED_FOLDER)
iraf.cd(TRIMMED_FOLDER)
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# I just want to remove the overscan region and trim the images
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = True
iraf.ccdproc.trim = True
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.biassec = "[308:384,1:1700]"
iraf.ccdproc.trimsec = "[1:300,1:1700]"

iraf.ccdproc.interactive = True
iraf.ccdproc.order = 6

# Do this for running the whole sample
#for i in obsnights:
#    iraf.ccdproc(os.path.join(TRIMMED_PATH, "night{0:d}/night{0:d}.*.fit".format(i)))
iraf.ccdproc("night1/night1.0*.fit")

night1/night1.001.fit:
night1/night1.002.fit:
night1/night1.003.fit:
night1/night1.004.fit:
night1/night1.005.fit:
night1/night1.006.fit:
night1/night1.007.fit:
night1/night1.008.fit:
night1/night1.009.fit:
night1/night1.010.fit:
night1/night1.011.fit:
night1/night1.012.fit:
night1/night1.013.fit:
night1/night1.014.fit:
night1/night1.015.fit:
night1/night1.016.fit:
night1/night1.017.fit:
night1/night1.018.fit:
night1/night1.019.fit:
night1/night1.020.fit:
night1/night1.021.fit:
night1/night1.022.fit:
night1/night1.023.fit:
night1/night1.024.fit:
night1/night1.025.fit:
night1/night1.026.fit:
night1/night1.027.fit:
night1/night1.028.fit:
night1/night1.029.fit:
night1/night1.030.fit:
night1/night1.031.fit:
night1/night1.032.fit:
night1/night1.033.fit:
night1/night1.034.fit:
night1/night1.035.fit:
night1/night1.036.fit:
night1/night1.037.fit:
night1/night1.038.fit:
night1/night1.039.fit:
night1/night1.040.fit:
night1/night1.041.fit:
night1/night1.042.fit:
night1/night1.043.fit:
night1/nigh

In [7]:
iraf.prows("night1/night1.001.fit", 100, 1600)

The image of the full chip shows a gradient across the chip over the spatial axis of around 20 counts. Hopefully this will be removed by the bias. The gradient also exists across the overscan region.

In [41]:
iraf.prows("night1/night1.001.fit", 100, 1600)

In [42]:
iraf.pcols("night1/night1.001.fit", 10, 290)

With the trimmed image, you basically see the spatial gradient as before. However, down the chip on the dispersion axis, 
there is very little gradient. Maybe of around 2 counts.

Now let's look at the differences between the bias frames of these objects.

In [6]:
ZEROED_FOLDER = "Modspec_Zeroproc"
iraf.cd(IMAGE_PATH)
shutil.copytree(TRIMMED_FOLDER, ZEROED_FOLDER)
iraf.cd(ZEROED_FOLDER)

NameError: name 'TRIMMED_FOLDER' is not defined

In [19]:
for i in obsnights:
    shutil.rmtree("night{0:d}/Biasdiffs/".format(i), ignore_errors=True)
    iraf.mkdir("night{0:d}/Biasdiffs/".format(i))
    with open(os.path.join(IMAGE_PATH, TRIMMED_FOLDER, "Night{0:d}_Biases.txt".format(i))) as biases:
        biaslist = biases.readlines()
        reference_index = 6
        ref_frame = biaslist[reference_index][:-1]
        ref_number = int(ref_frame[-7:-4])
        for j in xrange(len(biaslist)):
            bias_frame = biaslist[j][:-1]
            bias_number = int(bias_frame[-7:-4])
            iraf.imarith(bias_frame, "-", ref_frame, "night{0:d}/Biasdiffs/Biasdiff{1:d}{2:d}.fit".format(
                i, bias_number, ref_number)) 

I looked through the differences between the bias images for all of the nights. Some notable findings are:

* All nights have frames with transient diagonal structure in the bias images. It is not in phase between nights.

* Transient structure amplitude is small even on small scales. When running pcols over a very narrow column range to probe the amplitude of the structure, it was lost in the Poisson noise between the frames.

* On occasion, the diagonal structure can be irregular and fringy on certain frames. This level of fringiness is on the order of 5 counts. Less in other frames.

* Night 5 did not have frames with structure.

In [47]:
iraf.combine.combine = "average"
iraf.combine.reject = "minmax"
iraf.combine.scale = "none"
iraf.combine.nlow = 0
iraf.combine.nhigh = 1
iraf.combine.mclip = "yes"
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3

iraf.mkdir("Calibrations")
for i in obsnights:
    iraf.combine("Night{0:d}_Biases.txt".format(i), os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i)))

<function pyraf.iraffunctions.wrapper>

In [20]:
shutil.rmtree(os.path.join("Calibrations", "Biasdiffs"), ignore_errors=True)
iraf.mkdir(os.path.join("Calibrations", "Biasdiffs"))
ref_night = 6
reference_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(ref_night))
for i in obsnights:
    current_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
    iraf.imarith(current_bias, "-", reference_bias, os.path.join("Calibrations", "Biasdiffs", 
                                                                 "Biasdiffn{0:d}n{1:d}.fit".format(i, ref_night)))

In [23]:
difflist = glob.glob(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "Calibrations", "Biasdiffs", "Biasdiff*.fit"))
for diffimg in difflist:
    imgname = os.path.basename(diffimg)
    iraf.display(os.path.join("Calibrations", "Biasdiffs", imgname), 1, zscale=False, zrange=False, z1=-2, z2=2)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

z1=-2. z2=2.
Displaying Biasdiffn10n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn11n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn12n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn13n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn14n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn1n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn3n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn4n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn5n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn6n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn7n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn8n6.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiffn9n6.fit
Hit enter for next image.


There is additional structure between nights on the 0.3 count level. It also slopes by 0.3 counts over the spatial axis. The dispersion axis seems stable and well-behaved.

In [19]:
for i in obsnights:
    with open(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "Night{0:d}_Biases.txt".format(i))) as biases:
        biaslist = biases.readlines()
        night_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
        for j in xrange(len(biaslist)):
            bias_frame = os.path.join("..", TRIMMED_FOLDER, biaslist[j][:-1])
            bias_number = int(bias_frame[-7:-4])
            try:
                os.remove(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "night{0:d}".format(i), "Biasdiffs", 
                                       "Biasdiff{1:d}n{0:d}.fit".format(i, bias_number)))
            except OSError:
                pass
            iraf.imarith(bias_frame, "-", night_bias, os.path.join("night{0:d}".format(i), "Biasdiffs", 
                                   "Biasdiff{1:d}n{0:d}.fit".format(i, bias_number)))

In [33]:
nightno = 14
difflist = glob.glob(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "night{0:d}".format(nightno), "Biasdiffs", 
                                  "Biasdiff*n{0:d}.fit".format(nightno)))
for diffimg in difflist:
    imgname = os.path.basename(diffimg)
    iraf.display(os.path.join("night{0:d}".format(nightno), "Biasdiffs", imgname), 1, zscale=False, zrange=False, z1=-2, z2=2)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

z1=-2. z2=2.
Displaying Biasdiff46n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff47n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff48n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff49n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff50n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff51n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff52n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff53n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff54n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff55n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff56n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff57n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff58n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff59n14.fit
Hit enter for next image.
z1=-2. z2=2.
Displaying Biasdiff60n14.fit
Hit en

The fringes are still around. At this point, there doesn't seem to be an issue. I think at this point, there won't be an improvement in averaging things further.

In [32]:
biasdiffs = glob.glob(os.path.join(IMAGE_PATH, TRIMMED_FOLDER, "night*", "Biasdiffs"))
copylocs = map(lambda x: x.replace(TRIMMED_FOLDER, ZEROED_FOLDER), biasdiffs)
for src, dst in zip(biasdiffs, copylocs):
    shutil.copytree(src, dst)

In [17]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = True
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
#for i in obsnights:
#    iraf.ccdproc(os.path.join("night{0:d}".format(i), "night{0:d}.*.fit".format(i)), 
#                              zero=os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
iraf.ccdproc("night1/night1.0*.fit")

# Flat Fielding

Now it's time to look at the flat field exposures. Here I'll note the ratios between the fields.

In [55]:
FLATFIELD_FOLDER = "Modspec_Flatproc"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, ZEROED_FOLDER), os.path.join(IMAGE_PATH, FLATFIELD_FOLDER))
iraf.cd(FLATFIELD_FOLDER)

In [55]:
for i in obsnights:
    shutil.rmtree("night{0:d}/Flatratios/".format(i), ignore_errors=True)
    iraf.mkdir("night{0:d}/Flatratios/".format(i))
    with open(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Night{0:d}_Flats.txt".format(i))) as flats:
        flatlist = flats.readlines()
        reference_index = 6
        ref_frame = flatlist[reference_index][:-1]
        ref_number = int(ref_frame[-7:-4])
        # Need to scale the flats. I want to use imstat for this.
        ref_imstat_output = iraf.imstat(ref_frame+"[1:300,1:1200]", Stdout=1, fields="image,npix,mode")
        ref_imstat_dict = dict(zip(ref_imstat_output[0][1:].split(), ref_imstat_output[1].split()))
        ref_mode = float(ref_imstat_dict["MODE"])
        for j in xrange(len(flatlist)):
            flat_frame = flatlist[j][:-1]
            flat_number = int(flat_frame[-7:-4])
            flat_imstat_output = iraf.imstat(flat_frame+"[1:300,1:1200]", Stdout=1, fields="image,npix,mode")
            flat_imstat_dict = dict(zip(flat_imstat_output[0][1:].split(), flat_imstat_output[1].split()))
            flat_mode = float(flat_imstat_dict["MODE"])
            scaled_frame = flat_frame.replace(".{0:03d}.".format(flat_number), ".s{0:03d}.".format(flat_number))
            iraf.imarith(flat_frame, "*", ref_mode / flat_mode, scaled_frame)
            iraf.imarith(flat_frame, "/", ref_frame, "night{0:d}/Flatratios/Flatratio{1:d}{2:d}.fit".format(
                i, flat_number, ref_number)) 

In [54]:
nightno = 5
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Flatratio*.fit".format(nightno)))
for ratioimg in ratiolist:
    imgname = os.path.basename(ratioimg)
    iraf.display(os.path.join("night{0:d}".format(nightno), "Flatratios", imgname), 1, zscale=False, zrange=False, z1=0.9, 
                 z2=1.1)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

z1=0.9 z2=1.1
Displaying Flatratio3137.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3237.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3337.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3437.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3537.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3637.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3737.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3837.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio3937.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio4037.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio4137.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio4237.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio4337.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio4437.fit
Hit enter for next image.
z1=0.9 z2=1.1
Displaying Flatratio

In [76]:
nightno = 14
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Flatratio*.fit".format(nightno)))
current_path = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, current_path)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 1, 300, append=append, wy1=0.6, wy2=1.4)

There is a definite trend over many nights where the flat field lamp varies in brightness and temperature. The slope of the curve definitely correlates with the overall brightness of the lamp. When the lamp is brighter, it slopes up, when the lamp is fainter, it slopes down compared to a standard exposure.

As a result, it's preferable to fit the response function first, and then combine the flat fields.

In [54]:
iraf.twodspec()
iraf.longslit()

twodspec/:
 apextract/     longslit/
longslit/:
 aidpars@       deredden        identify        sarith          specplot
 autoidentify   dopcor          illumination    scopy           specshift
 background     extinction      lcalib          sensfunc        splot
 bplot          fceval          lscombine       setairmass      standard
 calibrate      fitcoords       reidentify      setjd           transform
 demos          fluxcalib       response        sflip


In [11]:
iraf.response.interactive = True
iraf.response.order = 11
iraf.response.low_reject = 3
iraf.response.high_reject = 3

for i in obsnights:
    with open(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Night{0:d}_Flats.txt".format(i))) as flats:
        flatlist = flats.readlines()
        for flatname in flatlist:
            flatname = flatname[:-1]
            flat_number = int(flatname[-7:-4])
            corrected_flat = flatname.replace(".{0:03d}.".format(flat_number), ".n{0:03d}.".format(flat_number))
            iraf.response(flatname, flatname, corrected_flat)

NameError: name 'FLATFIELD_FOLDER' is not defined

The flats should all be corrected and normalized now, and stored in .n???.fit files. Check to make sure that the flat fields are well-behaved.

In [12]:
nightno = 3
normed = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), 
                                "night{0:d}.n*.fit".format(nightno)))
current_path = iraf.pwd(Stdout=1)[0]
for norm in normed:
    if norm is normed[0]:
        append=False
    else:
        append=True
    iraf.pcols(norm, 1, 300, append=append, wy1=0.98, wy2=1.02)

The flat fields are fairly uniform and well-behaved up to pixel 1200, after which they get to be pretty ratty. Be wary of using the flatfield past that. Before that, the flatfields seem to be uniform down to 0.5%. Now let's combine these normalized flatfields.

In [ ]:
combine.reject = "avsigclip"
combine.scale = "mode"
combine.nlow = 1
combine.nhigh = 1
combine.nkeep = 1
combine.lsigma = 3
combine.hsigma = 3
combine.statsec="[1:300,1:1200]"

for i in obsnights:
    iraf.combine("Night{0:d}_Flats.txt".format(i), output=os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(i)))
        

In [13]:
shutil.rmtree(os.path.join("Calibrations", "Flatratios"), ignore_errors=True)
iraf.mkdir(os.path.join("Calibrations", "Flatratios"))
ref_night = 6
reference_flat = os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(ref_night))
for i in obsnights:
    current_flat = os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(i))
    iraf.imarith(current_flat, "/", reference_flat, os.path.join("Calibrations", "Flatratios", 
                                                                 "Flatration{0:d}n{1:d}.fit".format(i, ref_night)))

In [17]:
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Calibrations", "Flatratios", "Flatratio*.fit"))
current_path = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, current_path)
    iraf.display(imgname, 1, zscale=False, zrange=False, z1=0.95, z2=1.05)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")
    

z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration10n6.fit
Hit enter for next image.
z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration11n6.fit
Hit enter for next image.
z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration12n6.fit
Hit enter for next image.
z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration13n6.fit
Hit enter for next image.
z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration14n6.fit
Hit enter for next image.
z1=0.95 z2=1.05
Displaying ../../../../../home/gregory/SCIENCE/Binaries/Modspec/Modspec_Flatproc/Calibrations/Flatratios/Flatration1n6.fit
Hit enter for next image.
z1=0.

Nightly combined flats seem to differ from each other on a level of 0.1%, which is extremely small. I think we should just combine all flats into a master flat. Additionally, the differences in structure seen on the chip seemed to be small compared to the flat noise.

In [ ]:
iraf.combine(os.path.join("Calibrations", "Night*_Flat.fit"), output=os.path.join("Calibrations", "Master_Flat.fit"))

In [20]:
iraf.display(os.path.join("Calibrations", "Master_Flat.fit"), 1, zscale=False, zrange=False, z1=0.9, z2=1.1)

z1=0.9 z2=1.1


In [26]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = True
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    try:
        iraf.ccdproc("@Night{0:d}_Twilight.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    except iraf.IrafError:
        pass
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))

night1/night1.026.fit:
night1/night1.027.fit:
night1/night1.028.fit:
night1/night1.029.fit:
night1/night1.030.fit:
night1/night1.031.fit:
night1/night1.032.fit:
night1/night1.033.fit:
night1/night1.034.fit:
night1/night1.035.fit:
night1/night1.047.fit:
night1/night1.048.fit:
night1/night1.049.fit:
night1/night1.050.fit:
night1/night1.051.fit:
night1/night1.052.fit:
night1/night1.053.fit:
night1/night1.054.fit:
night1/night1.055.fit:
night1/night1.056.fit:
night1/night1.036.fit:
night1/night1.037.fit:
night1/night1.038.fit:
night1/night1.039.fit:
night1/night1.040.fit:
night1/night1.041.fit:
night1/night1.042.fit:
night1/night1.043.fit:
night1/night1.044.fit:
night1/night1.046.fit:


Killing IRAF task `ccdproc'


night1/night1.072.fit:
night1/night1.073.fit:
night1/night1.074.fit:
night1/night1.075.fit:
night1/night1.076.fit:
night1/night1.077.fit:
night1/night1.078.fit:
night1/night1.079.fit:
night1/night1.080.fit:
night1/night1.081.fit:
night1/night1.082.fit:
night1/night1.083.fit:
night1/night1.084.fit:
night1/night1.085.fit:
night1/night1.086.fit:
night1/night1.087.fit:
night1/night1.088.fit:
night1/night1.089.fit:
night1/night1.090.fit:
night1/night1.091.fit:
night1/night1.092.fit:
night1/night1.093.fit:
night1/night1.094.fit:
night1/night1.095.fit:
night1/night1.096.fit:
night1/night1.097.fit:
night1/night1.098.fit:
night1/night1.099.fit:
night1/night1.100.fit:
night1/night1.101.fit:
night1/night1.102.fit:
night1/night1.103.fit:
night1/night1.104.fit:
night1/night1.105.fit:
night1/night1.106.fit:
night1/night1.107.fit:
night1/night1.108.fit:
night1/night1.109.fit:
night1/night1.110.fit:
night1/night1.111.fit:
night1/night1.112.fit:
night1/night1.113.fit:
night1/night1.114.fit:
night1/nigh

night4/night4.184.fit:
night4/night4.185.fit:
night4/night4.186.fit:
night4/night4.187.fit:
night4/night4.188.fit:
night4/night4.189.fit:
night4/night4.190.fit:
night4/night4.191.fit:
night4/night4.192.fit:
night4/night4.193.fit:
night4/night4.194.fit:
night4/night4.195.fit:
night4/night4.196.fit:
night4/night4.197.fit:
night4/night4.198.fit:
night4/night4.199.fit:
night4/night4.200.fit:
night4/night4.201.fit:
night4/night4.202.fit:
night4/night4.203.fit:
night4/night4.204.fit:
night4/night4.205.fit:
night4/night4.206.fit:
night4/night4.207.fit:
night4/night4.208.fit:
night4/night4.209.fit:
night4/night4.210.fit:
night4/night4.211.fit:
night4/night4.212.fit:
night4/night4.213.fit:
night4/night4.214.fit:
night4/night4.215.fit:
night4/night4.216.fit:
night4/night4.217.fit:
night4/night4.218.fit:
night4/night4.219.fit:
night4/night4.220.fit:
night4/night4.221.fit:
night5/night5.001.fit:
night5/night5.002.fit:
night5/night5.003.fit:
night5/night5.004.fit:
night5/night5.005.fit:
night5/nigh

night6/night6.192.fit:
night6/night6.193.fit:
night6/night6.194.fit:
night6/night6.195.fit:
night6/night6.196.fit:
night6/night6.197.fit:
night6/night6.198.fit:
night6/night6.199.fit:
night6/night6.200.fit:
night6/night6.201.fit:
night6/night6.202.fit:
night6/night6.203.fit:
night6/night6.204.fit:
night6/night6.205.fit:
night6/night6.206.fit:
night6/night6.207.fit:
night6/night6.208.fit:
night6/night6.209.fit:
night6/night6.210.fit:
night6/night6.211.fit:
night6/night6.212.fit:
night6/night6.213.fit:
night6/night6.214.fit:
night6/night6.215.fit:
night6/night6.216.fit:
night6/night6.217.fit:
night6/night6.218.fit:
night6/night6.219.fit:
night6/night6.220.fit:
night6/night6.221.fit:
night6/night6.222.fit:
night6/night6.223.fit:
night6/night6.224.fit:
night6/night6.225.fit:
night6/night6.226.fit:
night6/night6.227.fit:
night6/night6.228.fit:
night6/night6.229.fit:
night6/night6.230.fit:
night6/night6.231.fit:
night6/night6.232.fit:
night6/night6.233.fit:
night6/night6.234.fit:
night6/nigh

...

night9/night9.071.fit:
night9/night9.072.fit:
night9/night9.073.fit:
night9/night9.074.fit:
night9/night9.075.fit:
night9/night9.076.fit:
night9/night9.077.fit:
night9/night9.078.fit:
night9/night9.079.fit:
night9/night9.080.fit:
night9/night9.081.fit:
night9/night9.082.fit:
night9/night9.083.fit:
night9/night9.084.fit:
night9/night9.085.fit:
night9/night9.086.fit:
night9/night9.087.fit:
night9/night9.088.fit:
night9/night9.089.fit:
night9/night9.090.fit:
night9/night9.091.fit:
night9/night9.092.fit:
night9/night9.093.fit:
night9/night9.094.fit:
night9/night9.095.fit:
night9/night9.096.fit:
night9/night9.097.fit:
night9/night9.098.fit:
night9/night9.099.fit:
night9/night9.100.fit:
night9/night9.101.fit:
night9/night9.102.fit:
night9/night9.103.fit:
night9/night9.104.fit:
night9/night9.105.fit:
night9/night9.106.fit:
night9/night9.107.fit:
night9/night9.108.fit:
night9/night9.109.fit:
night9/night9.110.fit:
night9/night9.111.fit:
night9/night9.112.fit:
night9/night9.113.fit:
night9/nigh

Killing IRAF task `ccdproc'


night10/night10.071.fit:
night10/night10.072.fit:
night10/night10.073.fit:
night10/night10.074.fit:
night10/night10.075.fit:
night10/night10.076.fit:
night10/night10.077.fit:
night10/night10.078.fit:
night10/night10.079.fit:
night10/night10.080.fit:
night10/night10.081.fit:
night10/night10.082.fit:
night10/night10.083.fit:
night10/night10.084.fit:
night10/night10.085.fit:
night10/night10.086.fit:
night10/night10.087.fit:
night10/night10.088.fit:
night10/night10.089.fit:
night10/night10.090.fit:
night10/night10.091.fit:
night10/night10.092.fit:
night10/night10.093.fit:
night10/night10.094.fit:
night10/night10.095.fit:
night10/night10.096.fit:
night10/night10.097.fit:
night10/night10.098.fit:
night10/night10.099.fit:
night10/night10.100.fit:
night10/night10.101.fit:
night10/night10.102.fit:
night10/night10.103.fit:
night10/night10.104.fit:
night10/night10.105.fit:
night10/night10.106.fit:
night10/night10.107.fit:
night10/night10.108.fit:
night10/night10.109.fit:
night10/night10.110.fit:


night11/night11.215.fit:
night11/night11.216.fit:
night11/night11.217.fit:
night11/night11.218.fit:
night11/night11.219.fit:
night11/night11.220.fit:
night11/night11.221.fit:
night11/night11.222.fit:
night11/night11.223.fit:
night11/night11.224.fit:
night11/night11.225.fit:
night11/night11.225.fit:
night11/night11.226.fit:
night11/night11.227.fit:
night11/night11.228.fit:
night11/night11.229.fit:
night11/night11.230.fit:
night11/night11.231.fit:
night12/night12.001.fit:
night12/night12.002.fit:
night12/night12.003.fit:
night12/night12.004.fit:
night12/night12.005.fit:
night12/night12.006.fit:
night12/night12.007.fit:
night12/night12.008.fit:
night12/night12.009.fit:
night12/night12.010.fit:
night11/night11.011.fit:
night11/night11.012.fit:
night11/night11.013.fit:
night11/night11.014.fit:
night11/night11.015.fit:
night11/night11.016.fit:
night11/night11.017.fit:
night11/night11.018.fit:
night11/night11.019.fit:
night11/night11.020.fit:
night11/night11.021.fit:
night11/night11.022.fit:


night13/night13.155.fit:
night13/night13.156.fit:
night13/night13.157.fit:
night13/night13.158.fit:
night13/night13.159.fit:
night13/night13.160.fit:
night13/night13.161.fit:
night13/night13.162.fit:
night13/night13.163.fit:
night13/night13.164.fit:
night13/night13.165.fit:
night13/night13.166.fit:
night13/night13.167.fit:
night13/night13.168.fit:
night13/night13.169.fit:
night13/night13.170.fit:
night13/night13.171.fit:
night13/night13.172.fit:
night13/night13.173.fit:
night13/night13.174.fit:
night13/night13.175.fit:
night13/night13.176.fit:
night13/night13.177.fit:
night13/night13.178.fit:
night13/night13.179.fit:
night13/night13.180.fit:
night13/night13.181.fit:
night13/night13.182.fit:
night13/night13.183.fit:
night13/night13.184.fit:
night13/night13.185.fit:
night13/night13.186.fit:
night13/night13.187.fit:
night13/night13.188.fit:
night13/night13.189.fit:
night13/night13.190.fit:
night13/night13.191.fit:
night13/night13.192.fit:
night13/night13.193.fit:
night13/night13.194.fit:


# Illumination


Get the illumination correction handled correctly now that the flat field is complete.

In [4]:
ILLUM_FOLDER = "Modspec_Illumproc"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER), os.path.join(IMAGE_PATH, ILLUM_FOLDER))
iraf.cd(ILLUM_FOLDER)

In order to do the illumination corrections, we have to look at the twilight exposures. These should now have been flatfield-corrected.

In [5]:
nightno = 13
scale_value = 1000.0
with open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(nightno))) as twilights:
    twilist = twilights.readlines()
    for twi in twilist:
        twiname = twi[:-1]
        twi_number = int(twiname[-7:-4])
        twi_imstat_output = iraf.imstat(twiname+"[10:290,100:1300]", Stdout=1, fields="image,npix,midpt,mode")
        twi_imstat_dict = dict(zip(twi_imstat_output[0][1:].split(), twi_imstat_output[1].split()))
        twi_mode = float(twi_imstat_dict["MIDPT"])
        scaled_frame = twiname.replace(".{0:03d}.".format(twi_number), ".s{0:03d}.".format(twi_number))
        try:
            os.remove(os.path.join(IMAGE_PATH, ILLUM_FOLDER, scaled_frame))
        except OSError:
            pass
        iraf.imarith(twiname, "*", scale_value / twi_mode, scaled_frame)
        print(twi_mode)
        if twi is twilist[0]:
            append = False
        else:
            append = True
        iraf.prows(scaled_frame, 100, 1300, append=append)

1436.0
437.0


Slope changes down the chip. So we'll have to correct them piece by piece.

In [134]:
for i in obsnights:
    try:
        twilights = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(i))) 
    except IOError:
        pass
    else:
        twilist = twilights.readlines()
        reference_index = -1
        ref_name = twilist[reference_index][:-1]
        ref_number = int(ref_name[-7:-4])
        scaled_ref = ref_name.replace(".{0:03d}.".format(ref_number), ".s{0:03d}.".format(ref_number))
        for twi in twilist:
            twiname = twi[:-1]
            twi_number = int(twiname[-7:-4])
            scaled_frame = twiname.replace(".{0:03d}.".format(twi_number), ".s{0:03d}.".format(twi_number))
            try:
                os.remove("night{0:d}/Flatratios/Twiratio{1:d}{2:d}.fit".format(i, twi_number, ref_number))
            except OSError:
                pass
            iraf.imarith(scaled_frame, "/", scaled_ref, "night{0:d}/Flatratios/Twiratio{1:d}{2:d}.fit".format(
                i, twi_number, ref_number)) 

In [37]:
nightno = 4
ratiolist = glob.glob(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "night{0:d}".format(nightno), "Flatratios", "Twiratio*.fit"))
currentpath = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(os.path.realpath(ratioimg), currentpath)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.prows(imgname, 400, 800, append=append, wy1=0.95, wy2=1.05)
    

In [7]:
nightno = 11
ratiolist = glob.glob(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Twiratio*.fit".format(nightno)))
currentpath = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, currentpath)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 220, 290, append=append, wy1=0.8, wy2=1.2)

Strange that the behavior of the twilights seems to be different along the dispersion axis. This may mean that the illumination changes somehow. It's strange. Hopefully it doesn't indicate that I messed up with the dome.

Twilight flats seem to vary by around 10% along the dispersion axis.

Along the spatial axis, twilights don't seem to vary at all. There are some differenes in the scaling, but generally they are consistently flat within a night.

In [19]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.weight = "median"
iraf.combine.blank = 1
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[10:290,1:1200]"

# Make twilight frames for each night
for i in obsnights:
    os.remove(os.path.join("Calibrations", "Night{0:d}_Twilight.fit".format(i)))
    iraf.combine("Night{0:d}_Twilight.txt".format(i), 
                 output=os.path.join("Calibrations", "Night{0:d}_Twilight.txt".format(i)))
        

Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have differe

In [47]:
nightno = 12
twitargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(i)))
twilist = twitargs.readlines()
ratiolist = map(lambda x: x.replace(".{0}.".format(x[-8:-5]), ".f{0}.".format(x[-8:-5])), twilist)
for ratioimg in ratiolist[1:2]:
    imgname = ratioimg[:-1]
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 130, 145, append=False, wy1=1.0, wy2=1.5)

In [43]:
# Now I want to do this for the Kepler target exposures to see if it matches.
for i in obsnights:
    keptargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)))
    for kep in keptargs:
        kepname = kep[:-1]
        kep_number = int(kepname[-7:-4])
        left_name = kepname+"[1:150,1:1700]"
        right_name = kepname+"[151:300,1:1700]"
        scaled_frame = kepname.replace(".{0:03d}.".format(kep_number), ".f{0:03d}.".format(kep_number))
        iraf.imarith(right_name, "/", left_name, scaled_frame) 
    keptargs.close()

In [52]:
nightno = 12
keptargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)))
keplist = keptargs.readlines()
ratiolist = map(lambda x: x.replace(".{0}.".format(x[-8:-5]), ".f{0}.".format(x[-8:-5])), keplist)
for ratioimg in ratiolist[5:6]:
    imgname = ratioimg[:-1]
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 100, 145, append=False, wy1=1.0, wy2=1.5)

This didn't work out well. There's way too much noise in these observations. I'll try co-adding all of the Kepler observations and then seeing if that helps tamp down the noise.

In [59]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[1:75,1:1200]"

for i in obsnights:
    os.remove(os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i)))
    iraf.combine("@Night{0:d}_KIC_Objects.txt".format(i), output=os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i)))
        

In [61]:
for i in obsnights:
    sky_image = os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i))
    left_name = sky_image+"[1:150,1:1700]"
    right_name = sky_image+"[151:300,1:1700]"
    scaled_frame = sky_image.replace("Sky", "Skyratio")
    iraf.imarith(right_name, "/", left_name, scaled_frame) 

In [ ]:
nightno = 4
iraf.pcols(os.path.join("Calibrations", "Night{0:d}_Skyratio.fit".format(nightno)), )

In [58]:
flatratios = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night*", "Flatratios"))
copylocs = map(lambda x: x.replace(FLATFIELD_FOLDER, ILLUM_FOLDER), flatratios)
for src, dst in zip(flatratios, copylocs):
    shutil.copytree(src, dst)

* Show that twilight exposures are the same long the dispersion axis.
* Compare twilights to sky values.
* Combine twilights.

In [44]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = True
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    try:
        iraf.ccdproc("@Night{0:d}_Twilight.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    except iraf.IrafError:
        pass
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))

night14/night14.s073.fit


In [139]:
# Create text files for object files.
for i in obsnights[2:3]:
    with open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Objects.txt".format(i))) as targets, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)), "w") as kics, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Standards.txt".format(i)), "w") as standards, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Arcs.txt".format(i)), "w") as arcs:
        for frame in targets:
            objpath = os.path.join(IMAGE_PATH, ILLUM_FOLDER, frame[:-1])
            targethdu = fits.open(objpath)
            objname = targethdu[0].header["OBJECT"]
            exptime = targethdu[0].header["EXPTIME"]
            targethdu.close()
            if objname.endswith("Arc"):
                targetfile = arcs
            elif objname.startswith("HD") or objname.startswith("BD") or objname.startswith("HIP"):
                targetfile = standards
            elif objname.startswith("KIC"):
                targetfile = kics
            else:
                print("Don't know file: {0}".format(objpath))
                continue
            targetfile.write(frame)

NameError: name 'ILLUM_FOLDER' is not defined

In [6]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.weight = "median"
iraf.combine.blank = 1
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[10:290,1:1200]"


os.remove(os.path.join("Calibrations", "Master_Twilight.fit"))
iraf.combine("@Twilights.txt", output=os.path.join("Calibrations", "Master_Twilight.fit"))
        

In [20]:
iraf.illum.interact = True
iraf.illum.nbins = 9
iraf.illum.low_reject = 3
iraf.illum.high_reject = 3
iraf.illum.order = 5

#os.remove(os.path.join("Calibrations", "Illum.fit"))
iraf.illum(os.path.join("Calibrations", "Master_Twilight.fit"), os.path.join("Calibrations", "Illum.fit"))

Determine illumination interactively for Calibrations/Master_Twilight.fit (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 1 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 2 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 3 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 4 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 5 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 6 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 7 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 8 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 9 (yes): 

The last bin looked kinda weird. But overall the twilight corrections seemed to be good!

In [24]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = False
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
# Only do the illumination correction.
iraf.ccdproc.illumcor = True
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))

# Calibration

In [4]:
CALIB_FOLDER = "Modspec_Calibration"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, ILLUM_FOLDER), os.path.join(IMAGE_PATH, CALIB_FOLDER))
iraf.cd(CALIB_FOLDER)

## Extract all of the spectra

In [11]:
iraf.apall.interactive = True
iraf.apall.find = True
iraf.apall.recenter = True
iraf.apall.resize = False
iraf.apall.edit = True
iraf.apall.trace = True
iraf.apall.extract = True
iraf.apall.review = True

iraf.apall.line = 148
iraf.apall.nsum = 10
iraf.apall.width = 24
iraf.apall.lower = -12
iraf.apall.upper = 12
iraf.apall.resize = False

iraf.apall.b_sample = "-100:-30,30:100"
iraf.apall.b_naver = -100
iraf.apall.b_funct = "chebyshev"
iraf.apall.b_order = 1
iraf.apall.b_high_rej = 3
iraf.apall.b_niter = 5
iraf.apall.b_grow = 1

iraf.apall.t_nsum = 10
iraf.apall.t_step = 10
iraf.apall.t_funct = "spline3"
iraf.apall.t_order = 2
iraf.apall.t_niter = 1

iraf.background = "fit"
iraf.apall.weights = "none"
iraf.apall.clean = False
iraf.apall.format = "multispec"
iraf.apall.extras = True

In [5]:
# These four files need to be existing in order to generate the other files:
# Night12_Standards.txt
# Night12_KIC_Objects.txt
# Night12_Standards_Arcs.txt
# Night12_KIC_Objects_Arcs.txt

def compact_standard(filename):
    '''Compactify a filename.'''
    compact = os.path.splitext(os.path.splitext(os.path.basename(filename))[0])[0].replace(".", "").replace("night","n")
    return compact

# These are for simple file naming. Just format and go!
# Examples of the file types are given above the template
obj_types = ("Standards", "KIC_Objects")
arctypes = ["ne", "xe", "ar"]
# raw_target_template.format(12, objtypes[0]) -> Night12_Standards.txt
raw_target_template = "Night{0:d}_{1}.txt"
# combined_target_template.format(12, objtypes[0]) -> Night12_Standards_Combined.txt
combined_target_template = "Night{0:d}_{1}_Combined.txt"
# extracted_target_template.format(12, objtypes[0]) -> Night12_Standards_Extracted.txt
extracted_target_template = "Night{0:d}_{1}_Extracted.txt"
# calibration_template.format(12, objtypes[0]) -> Night12_Standards_Calib.txt
calibrated_target_template = "Night{0:d}_{1}_Calib.txt"
# calibration_template.format(12, objtypes[0], arctypes[2].capitalize()) -> Night12_Standards_Ar.txt
calibration_template = "Night{0:d}_{1}_{2}.txt"
# repeat_calibration_template.format(12, objtypes[0], arctypes[2].capitalize()) -> Night12_Standards_Repeat_Ar.txt
repeat_calibration_template = "Night{0:d}_{1}_Repeat_{2}.txt"
# subtracted_calibration_template.format(12, objtypes[0], arctypes[2].capitalize() -> Night12_Standards_Ar_Subtracted.txt)
subtracted_calibration_template = "Night{0:d}_{1}_{2}_Subtracted.txt"
# fullspec_template.format(12, objtypes[0]) -> Night12_Standards_Fullspec.txt
fullspec_template = "Night{0:d}_{1}_Fullspec.txt"
# arc_template.format(12, objtypes[0]) -> Night12_Standards_Arcs.txt
arc_template = "Night{0:d}_{1}_Arcs.txt"
# extracted_arc_template.format(12, objtypes[0]) -> Night12_Standards_Arcs_Extracted.txt
extracted_arc_template = "Night{0:d}_{1}_Arcs_Extracted.txt"
# fxcor_flexure_template.format(12, objtypes[0]) -> Night12_Standards_Cor_Base.txt
fxcor_flexure_template = "Night{0:d}_{1}_Cor_Base.txt"
# arc_multitrace_template.format(12, objtypes[0], 121) -> Night12_Standards_Arc121.txt
arc_multitrace_template = "Night{0:d}_{1}_Arc{2}.txt"
# fxcor_multitrace_template.format(12, objtypes[0], 121) -> Night12_Standards_Arc121_FXcor.txt
fxcor_multitrace_template = "Night{0:d}_{1}_Arc{2}_FXcor.txt"
# target_cor_template.format(12, objtypes[0], compact_standard("night12/night12.c121.ms.fits").upper()) -> Night12_Standards_Cor_N12C121.txt
target_cor_template = "Night{0:d}_{1}_Cor_{2}.txt"

In [65]:
for n in obsnights:
    img_count = 1
    for targ in obj_types:
        raw_target_filelist = raw_target_template.format(n, targ)
        combined_target_filelist = combined_target_template.format(n, targ)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_target_filelist), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, combined_target_filelist), "w") as newfile:
                for imageline in oldfile:
                    images = imageline[:-1].split(" ")
                    if len(images) == 1:
                        newimage = images[0]
                    elif len(images) > 1:
                        newimage = images[0].replace(images[0][-7:-4],"d{0:02d}".format(img_count))
                        try:
                            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newimage))
                        except OSError:
                            pass
                        iraf.imcombine(",".join(images), newimage)
                        img_count = img_count + 1
                    newfile.write(newimage+"\n")
                        
                
        extracted_target_filelist = extracted_target_template.format(n, targ)
        # Make the filenames for the extracted objects
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, combined_target_filelist), 'r') as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), 'w') as newfile:
                for oldname in oldfile:
                    newname = oldname.replace(".fit", ".ms.fits")
                    newfile.write(newname)
        iraf.apall("@"+combined_target_filelist, output="@"+combined_target_filelist, intera="no")


Sep 19 20:14: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night4/night4.110.fit
  night4/night4.111.fit

  Output image = night4/night4.d01.fit, ncombine = 2

Sep 19 20:14: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night4/night4.205.fit
  night4/night4.206.fit

  Output image = night4/night4.d02.fit, ncombine = 2
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.078.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.079.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.082.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.083.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.086.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.087.ms already exists
Sep 19 20:14: EXTRACT - Output spectrum night4/night4.090.ms already exists
Sep 19 20:14: EX

In [37]:
# Apall has issues running. However, what happens afterward should be documented.
iraf.apall("@Night1_Standards.txt")

Recenter apertures for night1/night1.072?Resize apertures for night1/night1.072?Edit apertures for night1/night1.072?

     aperture = 1  beam = 1  center = 149.51  low = -3.36  upper = 3.03
Invalid or unrecognized command

       		 APEXTRACT CURSOR KEY SUMMARY

?  Print help             j  Set beam number        u  Set upper limit(s)
a  Toggle all flag        l  Set lower limit(s)     w  Window graph
b  Set background(s)      m  Mark aperture          y  Y level limit(s)
c  Center aperture(s)     n  New uncentered ap.     z  Resize aperture(s)
d  Delete aperture(s)     o  Order ap. numbers      I  Interrupt
e  Extract spectra        q  Quit                   +  Next aperture
f  Find apertures         r  Redraw graph           -  Previous aperture
g  Recenter aperture(s)   s  Shift aperture(s)      .  Nearest aperture
i  Set aperture ID        t  Trace aperture(s)      

       		 APEXTRACT COLON COMMAND SUMMARY

:apertures      :center         :npeaks         :show           :t_width
:apidtable      :clean          :nsubaps        :skybox         :threshold
:avglimits      :database       :nsum           :t_function     :title
:b_function     :extras         :order          :t_grow         :ulimit
:b_gr

Trace apertures for night1/night1.072?Fit traced positions for night1/night1.072 interactively?Fit curve to aperture 1 of night1/night1.072 interactivelyWrite apertures for night1/night1.072 to databaseExtract aperture spectra for night1/night1.072?Review extracted spectra from night1/night1.072?Clobber existing output image night1/night1.072.ms?

Aug 24 13:16: EXTRACT - Output spectrum night1/night1.072.ms already exists


Recenter apertures for night1/night1.075?Resize apertures for night1/night1.075?Edit apertures for night1/night1.075?

     aperture = 1  beam = 1  center = 149.31  low = -3.35  upper = 2.72
Aperture (1) =      aperture = 1  beam = 1  center = 149.31  low = -3.35  upper = 2.72
     aperture = 1  beam = 1  center = 149.31  low = -3.35  upper = 2.72
Invalid or unrecognized command

Killing IRAF task `apall'


KeyboardInterrupt: 

In [4]:
# Stuff that's going on
Calib_Night = 12

In [7]:
obsnights[10]

12

## Wavelength-Calibrate Spectra

In [7]:
xc=(0, 0, 0)
nc=(0, 109/255.0, 219/255.0)
ac=(219/255.0, 209/255.0, 0)
fc = (182/255.0, 219/255.0, 255/255.0)

In [26]:
# Generate the Calibration Spectra
iraf.combine.combine = "average"
iraf.combine.scale = "mode"
iraf.combine.weight = "mode"
iraf.combine.reject = "avsigclip"
iraf.combine.mclip = "yes"
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec = "[1:300,900:1100]"
for n in obsnights:
    for spec in arctypes:
        combofile = "Night{0:d}_{1}.txt".format(n, spec.capitalize())
        outputfile = os.path.join("Calibrations", "Night{0:d}_{1}.fit".format(n, spec.capitalize()))
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, outputfile))
        except OSError:
            pass
        iraf.combine("@"+combofile, outputfile)

In [66]:
# Extract calibration spectra using object and standard traces.
for n in obsnights:
    for targclass in obj_types:
        raw_target_filelist = combined_target_template.format(n, targclass)
        extracted_target_filelist = extracted_target_template.format(n, targclass)
        for spec in arctypes:
            extracted_calib_filelist = calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'w') as newfile:
                    for oldname in oldfile:
                        # turn night12.212.ms.fits to night12.ar212.ms.fits
                        nightname = "night{0:d}.".format(n)
                        newname = oldname.replace(nightname, nightname+spec)
                        newfile.write(newname)
            # Since apall can't deal with a single input, I'll loop through the name files manually with python instead 
            # of just creating a file with the input spectrum repeating.
            apall_repeat_file = repeat_calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, apall_repeat_file), 'w') as newfile:
                    for oldname in oldfile:
                        # Just write the master Calibration file over and over.
                        newname = os.path.join("Calibrations", "Night{0:d}_{1}.fit\n".format(n, spec.capitalize()))
                        newfile.write(newname)
            iraf.apall("@"+apall_repeat_file, out="@"+extracted_calib_filelist, ref="@"+raw_target_filelist, recen=False, 
                        trace=False, back="none", intera=False)
            print "Things Extracted."
            # Subtract the continuum from the lines.
            subtracted_calib_filelist = subtracted_calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, subtracted_calib_filelist), 'w') as newfile:
                    for oldname in oldfile:
                        # turn night12.ar212.ms.fits to night12.sar212.ms.fits
                        newname = oldname.replace(spec, "s"+spec)
                        newfile.write(newname)
                        try:
                            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                            print "Removed " + newname
                        except OSError:
                            pass
            iraf.continuum.func = "chebyshev"
            iraf.continuum.order = 15
            iraf.continuum.high_rej = 3
            iraf.continuum.low_rej = 0
            # Continuum isn't happy with empty files. So ignore this if it's empty.
            try:
                iraf.continuum("@"+extracted_calib_filelist, "@"+subtracted_calib_filelist, intera="no")
            except iraf.IrafError:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as infile:
                    contents = infile.readlines()
                    if not contents:
                        pass
                    else:
                        raise

            # Now reidentify the lines
            iraf.reidentify(os.path.join("calib_test", "{0}spec".format(spec)), "@"+subtracted_calib_filelist, intera="no")
        
# Combine the line identifications and refit the wavelength solution.
        fullspec_filelist = fullspec_template.format(n, targclass)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, 
                               subtracted_calibration_template.format(n, targclass, arctypes[2].capitalize())), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "w") as newfile:
            for oldname in oldfile:
                # turn night12.ar212.ms.fits to night12.full212.ms.fits
                newname = oldname.replace("sar", "full")
                shutil.copy(os.path.join(IMAGE_PATH, CALIB_FOLDER, oldname[:-1]), 
                            os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                newfile.write(newname)
        # Read in all of the features
        full_spec_table = []
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "r") as fullspecs:
            for fullimg in fullspecs:
                fulldb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
                for spec in ["ne", "ar", "xe"]:
                    specimg = fullimg.replace("full", "s"+spec)
                    specdb, ext = os.path.splitext(os.path.join("database", "id"+specimg))
                    # Read in the entry.
                    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specdb)) as specdata:
                        spec_fullfile = specdata.read()
                    spec_features = spec_fullfile[spec_fullfile.rindex("begin"):]
                    spec_length_line_start = spec_features.index("features")
                    spec_length_line_end = spec_features.index("\n", spec_length_line_start)
                    spec_numlines = int(spec_features[spec_length_line_start:spec_length_line_end].split("\t")[1])
                    spec_table_start = spec_length_line_end+1
                    # Subtract two because there is a trailing tab before "function"
                    spec_table_end = spec_features.index("function")-2
                    spec_table = spec_features[spec_table_start:spec_table_end].split("\n")
                    full_spec_table = full_spec_table + spec_table
                # Now combine them
                full_numlines = len(full_spec_table)
                full_feat_table = Table.read(full_spec_table, format="ascii.fixed_width_no_header", 
                                            names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                            col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
                full_feat_table["Count"] = np.arange(len(full_feat_table))
                full_feat_table.sort("Pixel")
                sorted_table = [full_spec_table[i] for i in full_feat_table["Count"]]
                new_feature_table = "\n".join(sorted_table)

                # Now let's piece together the new file. First make the time comment.
                a = datetime.now()
                comment_line = "# " + a.strftime("%a %H:%M:%S %d-%b-%Y") + "\n"
                # Then make the header:
                spec_head_start = 0
                # Note that this includes the leading tab character in the header, not as part of the "feature" line.
                spec_head_end = spec_length_line_start
                spec_header = spec_features[spec_head_start:spec_head_end]
                full_header = spec_header.replace("s"+spec, "full")
                # Now the feature line will be added on.
                full_feature_line = "features\t{0:d}\n".format(full_numlines)
                # Lastly we want the footer, which doesn't actually contain any useful information, but we will include.
                footer = spec_features[spec_table_end:]
                # Now add them all together!
                fullfile = comment_line + full_header + full_feature_line + new_feature_table + footer
        
                # Write the result to a file.
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fulldb), 'a') as fulldata:
                    fulldata.write(fullfile)
# Apply the wavelength solution to the full spectra.
        calibrated_target_filelist = calibrated_target_template.format(n, targclass)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist), "w") as newfile:
            for oldname in oldfile:
                # turn night12.212.ms.fits to night12.c212.ms.fits
                nightstr = "night{0:d}.".format(n)
                newname = oldname.replace(nightstr, nightstr+"c")
                newfile.write(newname)
        # I don't know why this isn't working. It says that the objects in the references list aren't real reference spectra.
        # Just set all of the spectra manually.
        # iraf.refspec("@"+extracted_target_filelist, references="@"+fullspec_filelist, confirm=True, select="match")
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist)) as fullspec, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist)) as stand:
                for ref, obj in zip(fullspec, stand):
                    print obj[:-1]
                    iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist)) as caltarg:
            for targ in caltarg:
                try:
                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, targ[:-1]))
                except OSError:
                    pass
        iraf.dispcor("@"+extracted_target_filelist, "@"+calibrated_target_filelist, linearize=True)
    
# Extract standard arc spectra
    raw_standard_file = combined_target_template.format(n, obj_types[0])
    standard_arcfile = arc_template.format(n, obj_types[0])
    extracted_standard_arcfile = extracted_arc_template.format(n, obj_types[0])
    print (standard_arcfile, raw_standard_file)
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_arcfile), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile), 'w') as newfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_standard_file), 'r') as companionfile:
        for oldname, compname in zip(oldfile, companionfile):
            standnum = compname[-8:-5]
            arcnum = oldname[-8:-5]
            newname = oldname.replace(".fit", ".ms.fits").replace(arcnum, arcnum+"t"+standnum)
            newfile.write(newname)
    iraf.apall("@"+standard_arcfile, out="@"+extracted_standard_arcfile, 
               ref="@"+raw_standard_file, recen=False, trace=False, back="none", intera=False)
# Cross-correlate extracted arc spectra with Argon trace to get flexure correction.
    iraf.fxcor.continuum = "both"
    iraf.fxcor.filter = "both"
    iraf.fxcor.pixcorr = "yes"
    iraf.fxcor.function = "gaussian"
    iraf.fxcor.observatory = "kpno"
    
    iraf.continpars.c_inter = True
    iraf.continpars.order = 20
    iraf.continpars.low_rej = 0
    iraf.continpars.high_rej = 2
    iraf.continpars.nitera = 10
    iraf.continpars.grow = 1
    
    iraf.filtpars.f_type = "square"
    iraf.filtpars.cuton = 30
    iraf.filtpars.cutoff = 1000
    
    fxcor_flexure_output = fxcor_flexure_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output), 'w') as newfile:
        for oldname in oldfile:
            newname = os.path.splitext(os.path.splitext(oldname)[0])[0]+"\n"
            newfile.write(newname)
            
    standard_calibration_argon = calibration_template.format(n, obj_types[0], arctypes[2].capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile)) as arcs, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_calibration_argon)) as calibs, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output)) as corfile:
            for arc, ar, cor in zip(arcs, calibs, corfile):
                if arc[-12:-9] != ar[-12:-9]:
                    print "{0} does not correctly trace night{1}.{2}.ms.fits".format(arc[:-1], n, ar[-12:-9])
                iraf.fxcor(arc[:-1], ar[:-1], out=cor[:-1], interact="no")

# Apply flexure correction.
    standard_file = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output)) as exarc_file, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_file)) as standards:
        shiftnums = []
        for arcbase, standimage in zip(exarc_file, standards):
            shiftfile = arcbase[:-1] + ".txt"
            shift_table = Table.read(shiftfile, format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                             header_start=13, guess=False, 
                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                    'VOBS', 'VREL', 'VHELIO', 'VERR'])
            shift = shift_table["SHIFT"][-1]
            
            # This is just to make sure that the standards and arcs match up. It's possible some weird things go on.
            objlabel = iraf.hedit(standimage[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            arclabel = shift_table["OBJECT"][-1]
            if objlabel + "_Arc" != arclabel:
                print "{0} does not match arc {1}".format(standimage[:-1], shift_table["IMAGE"][-1])
                print "{0} is not the arc of {1}".format(arclabel, objlabel)
                continue

            iraf.hedit(standimage[:-1], "CRPIX1", "(1-{0:g})".format(shift), verify=False)
            print "Corrected " + standimage[:-1]
    
# Get trace from (???) for Kepler arc spectra and Argon calibration.
    raw_kic_file = combined_target_template.format(n, obj_types[1])
    extracted_kic_file = extracted_target_template.format(n, obj_types[1])
    kic_arcfile = arc_template.format(n, obj_types[1])
    extracted_kic_arcfile = extracted_arc_template.format(n, obj_types[1])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcfile), 'r') as arcs:
        total_shifts = []
        for arc in arcs:
            arcnum = arc[-8:-5]
            output_traces = arc_multitrace_template.format(n, obj_types[1], arcnum)
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_kic_file), 'r') as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'w') as newfile:
                for oldname in oldfile:
                    kicnum = oldname[-8:-5]
                    newname = oldname.replace(".fit", ".ms.fits").replace(kicnum, arcnum+"t"+kicnum)
                    newfile.write(newname)
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_kic_file), 'r') as kics, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as outputs:
                    for kic, out in zip(kics, outputs):
                        iraf.apall(arc[:-1], out=out[:-1], ref=kic[:-1], recen=False, trace=False, back="none", 
                                   intera=False)
# Measure flexure throughout the night. (Output plots)
            fxcor_base = fxcor_multitrace_template.format(n, obj_types[1], arcnum)
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base), 'w') as newfile:
                for oldname in oldfile:
                    # night12.187t186.ms.fits -> night12.187t186
                    newname = os.path.splitext(os.path.splitext(oldname)[0])[0]+"\n"
                    newfile.write(newname)
            iraf.fxcor.continuum = "both"
            iraf.fxcor.filter = "both"
            iraf.fxcor.pixcorr = "yes"
            iraf.fxcor.function = "gaussian"
            iraf.fxcor.observatory = "kpno"
    
            iraf.continpars.c_inter = True
            iraf.continpars.order = 20
            iraf.continpars.low_rej = 0
            iraf.continpars.high_rej = 2
            iraf.continpars.nitera = 10
            iraf.continpars.grow = 1
    
            iraf.filtpars.f_type = "square"
            iraf.filtpars.cuton = 30
            iraf.filtpars.cutoff = 1000
    
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base)) as corfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces)) as arcfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, 
                                   calibration_template.format(n, obj_types[1], arctypes[2].capitalize())), 'r') as tempfile:
                arcshifts = []
                for corbase, trace, template in zip(corfile, arcfile, tempfile):
                    iraf.fxcor(trace[:-1], template[:-1], out=corbase[:-1], interact="no")
                               
                # Measure scatter in flexure
                    shiftfile = corbase[:-1] + ".txt"
                    shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile), 
                                             format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                             header_start=13, guess=False, 
                                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                                    'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
                    shift = shift_table["SHIFT"][-1]
                    arcshifts.append(shift)
    
            total_shifts.append(arcshifts)
    shift_array = np.array(total_shifts)
    flexures = np.mean(shift_array, axis=1)
# Interpolate and apply flexure correction to targets 
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcfile)) as arcs:
        arctimes = []
        for arc in arcs:
            hdulist = fits.open(arc[:-1])
            arctimes.append(hdulist[0].header["JD"])
            hdulist.close()
    times = np.array(arctimes)
    flex_interp = interp1d(times, flexures)
    
    calibrated_kic_file = calibrated_target_template.format(n, obj_types[1])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_kic_file)) as kics:
        for kicobj in kics:
            hdulist = fits.open(kicobj[:-1])
            try:
                flexcorr = flex_interp(hdulist[0].header["JD"])
            except ValueError:
                # I accidentally observed the object before the arc. So manually set the flexure to that measured by the arc.
                if kicobj[:-1] == "night7/night7.c073.ms.fits":
                    flexcorr = flex_interp.y[0]
                    
            hdulist.close()
            
            iraf.hedit(kicobj[:-1], "CRPIX1", "(1-{0})".format(flexcorr), verify=False)
            print "Corrected " + kicobj[:-1]
# Get J-K for Kepler targets
# Match Kepler target with standard.
# Cross-correlate Kepler target with standard (check with other standards)

Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne078.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne079.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne082.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne083.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne086.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne087.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne090.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne091.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne094.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne095.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne098.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne099.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne102.ms a

Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe204.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xed02.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe209.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe210.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe213.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe214.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe217.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe218.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.xe221.ms already exists
Things Extracted.
Removed night4/night4.sxe078.ms.fits

Removed night4/night4.sxe079.ms.fits

Removed night4/night4.sxe082.ms.fits

Removed night4/night4.sxe083.ms.fits

Removed night4/night4.sxe086.ms.fits

Removed night4/night4.sxe087.ms.fits

Removed night4/night4.sxe090.ms.fits

Removed night4

night4/night4.083.ms.fits
night4/night4.083.ms.fits,REFSPEC1: night4/night4.full083.ms.fits -> night4/night4.full083.ms.fits
night4/night4.083.ms.fits updated
night4/night4.086.ms.fits
night4/night4.086.ms.fits,REFSPEC1: night4/night4.full086.ms.fits -> night4/night4.full086.ms.fits
night4/night4.086.ms.fits updated
night4/night4.087.ms.fits
night4/night4.087.ms.fits,REFSPEC1: night4/night4.full087.ms.fits -> night4/night4.full087.ms.fits
night4/night4.087.ms.fits updated
night4/night4.090.ms.fits
night4/night4.090.ms.fits,REFSPEC1: night4/night4.full090.ms.fits -> night4/night4.full090.ms.fits
night4/night4.090.ms.fits updated
night4/night4.091.ms.fits
night4/night4.091.ms.fits,REFSPEC1: night4/night4.full091.ms.fits -> night4/night4.full091.ms.fits
night4/night4.091.ms.fits updated
night4/night4.094.ms.fits
night4/night4.094.ms.fits,REFSPEC1: night4/night4.full094.ms.fits -> night4/night4.full094.ms.fits
night4/night4.094.ms.fits updated
night4/night4.095.ms.fits
night4/night4.095.ms

night4/night4.c094.ms.fits: ap = 1, w1 = 4346.168, w2 = 6063.122, dw = 1.010567, nw = 1700
night4/night4.095.ms.fits: REFSPEC1 = 'night4/night4.full095.ms.fits 1.'
night4/night4.c095.ms.fits: ap = 1, w1 = 4346.163, w2 = 6063.127, dw = 1.010573, nw = 1700
night4/night4.098.ms.fits: REFSPEC1 = 'night4/night4.full098.ms.fits 1.'
night4/night4.c098.ms.fits: ap = 1, w1 = 4346.192, w2 = 6063.106, dw = 1.010544, nw = 1700
night4/night4.099.ms.fits: REFSPEC1 = 'night4/night4.full099.ms.fits 1.'
night4/night4.c099.ms.fits: ap = 1, w1 = 4346.174, w2 = 6063.116, dw = 1.010561, nw = 1700
night4/night4.102.ms.fits: REFSPEC1 = 'night4/night4.full102.ms.fits 1.'
night4/night4.c102.ms.fits: ap = 1, w1 = 4346.182, w2 = 6063.113, dw = 1.010554, nw = 1700
night4/night4.103.ms.fits: REFSPEC1 = 'night4/night4.full103.ms.fits 1.'
night4/night4.c103.ms.fits: ap = 1, w1 = 4346.168, w2 = 6063.138, dw = 1.010577, nw = 1700
night4/night4.106.ms.fits: REFSPEC1 = 'night4/night4.full106.ms.fits 1.'
night4/night4.c1

Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne172.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne173.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne175.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne176.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne177.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne178.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne179.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne181.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne182.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne183.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne184.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne185.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.ne186.ms a

...

Corrected night4/night4.c120.ms.fits
night4/night4.c123.ms.fits,CRPIX1: 1. -> 1.438
night4/night4.c123.ms.fits updated
Corrected night4/night4.c123.ms.fits
night4/night4.c124.ms.fits,CRPIX1: 1. -> 1.486
night4/night4.c124.ms.fits updated
Corrected night4/night4.c124.ms.fits
night4/night4.c127.ms.fits,CRPIX1: 1. -> 1.483
night4/night4.c127.ms.fits updated
Corrected night4/night4.c127.ms.fits
night4/night4.c128.ms.fits,CRPIX1: 1. -> 1.597
night4/night4.c128.ms.fits updated
Corrected night4/night4.c128.ms.fits
night4/night4.c131.ms.fits,CRPIX1: 1. -> 1.586
night4/night4.c131.ms.fits updated
Corrected night4/night4.c131.ms.fits
night4/night4.c132.ms.fits,CRPIX1: 1. -> 1.594
night4/night4.c132.ms.fits updated
Corrected night4/night4.c132.ms.fits
night4/night4.c135.ms.fits,CRPIX1: 1. -> 1.512
night4/night4.c135.ms.fits updated
Corrected night4/night4.c135.ms.fits
night4/night4.c137.ms.fits,CRPIX1: 1. -> 1.676
night4/night4.c137.ms.fits updated
Corrected night4/night4.c137.ms.fits
night4/nigh

Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t169.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t170.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t171.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t172.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t173.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t175.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t176.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t177.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t178.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t179.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t181.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum night4/night4.146t182.ms already exists
Sep 19 20:17: EXTRACT - Output spectrum 

Sep 19 20:18: EXTRACT - Output spectrum night4/night4.159t188.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.159t189.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t141.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t142.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t143.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t144.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t145.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t147.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t148.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t149.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t150.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.166t151.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum 

Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t158.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t160.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t161.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t162.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t163.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t164.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t165.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t167.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t168.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t169.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t170.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum night4/night4.180t171.ms already exists
Sep 19 20:18: EXTRACT - Output spectrum 

Corrected night4/night4.c164.ms.fits
night4/night4.c165.ms.fits,CRPIX1: 1. -> 1.650104
night4/night4.c165.ms.fits updated
Corrected night4/night4.c165.ms.fits
night4/night4.c167.ms.fits,CRPIX1: 1. -> 1.640326
night4/night4.c167.ms.fits updated
Corrected night4/night4.c167.ms.fits
night4/night4.c168.ms.fits,CRPIX1: 1. -> 1.623397
night4/night4.c168.ms.fits updated
Corrected night4/night4.c168.ms.fits
night4/night4.c169.ms.fits,CRPIX1: 1. -> 1.605046
night4/night4.c169.ms.fits updated
Corrected night4/night4.c169.ms.fits
night4/night4.c170.ms.fits,CRPIX1: 1. -> 1.588514
night4/night4.c170.ms.fits updated
Corrected night4/night4.c170.ms.fits
night4/night4.c171.ms.fits,CRPIX1: 1. -> 1.57347
night4/night4.c171.ms.fits updated
Corrected night4/night4.c171.ms.fits
night4/night4.c172.ms.fits,CRPIX1: 1. -> 1.55674
night4/night4.c172.ms.fits updated
Corrected night4/night4.c172.ms.fits
night4/night4.c173.ms.fits,CRPIX1: 1. -> 1.539348
night4/night4.c173.ms.fits updated
Corrected night4/night4.c1

In [33]:
print iraf.hedit(standimage[:-1], "OBJECT", ".", Stdout=1)

[]


Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne076.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne079.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne080.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne083.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne084.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne087.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne088.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne091.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne092.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne095.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne096.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ne099.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum 

Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar132.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar135.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar136.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar139.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar140.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar205.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar206.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar209.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar210.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar213.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar214.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.ar217.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum 


Removed night12/night12.sxe128.ms.fits

Removed night12/night12.sxe131.ms.fits

Removed night12/night12.sxe132.ms.fits

Removed night12/night12.sxe135.ms.fits

Removed night12/night12.sxe136.ms.fits

Removed night12/night12.sxe139.ms.fits

Removed night12/night12.sxe140.ms.fits

Removed night12/night12.sxe205.ms.fits

Removed night12/night12.sxe206.ms.fits

Removed night12/night12.sxe209.ms.fits

Removed night12/night12.sxe210.ms.fits

Removed night12/night12.sxe213.ms.fits

Removed night12/night12.sxe214.ms.fits

Removed night12/night12.sxe217.ms.fits

Removed night12/night12.sxe218.ms.fits

Removed night12/night12.sxe221.ms.fits

Removed night12/night12.sxe222.ms.fits

Removed night12/night12.sxe225.ms.fits

Removed night12/night12.sxe226.ms.fits

Removed night12/night12.sxe229.ms.fits

Removed night12/night12.sxe230.ms.fits

Removed night12/night12.sxe233.ms.fits

Removed night12/night12.sxe234.ms.fits

Removed night12/night12.sxe237.ms.fits

night12/night12.076.ms.fits
night12/nig

night12/night12.225.ms.fits
night12/night12.225.ms.fits,REFSPEC1: night12/night12.full225.ms.fits -> night12/night12.full225.ms.fits
night12/night12.225.ms.fits updated
night12/night12.226.ms.fits
night12/night12.226.ms.fits,REFSPEC1: night12/night12.full226.ms.fits -> night12/night12.full226.ms.fits
night12/night12.226.ms.fits updated
night12/night12.229.ms.fits
night12/night12.229.ms.fits,REFSPEC1: night12/night12.full229.ms.fits -> night12/night12.full229.ms.fits
night12/night12.229.ms.fits updated
night12/night12.230.ms.fits
night12/night12.230.ms.fits,REFSPEC1: night12/night12.full230.ms.fits -> night12/night12.full230.ms.fits
night12/night12.230.ms.fits updated
night12/night12.233.ms.fits
night12/night12.233.ms.fits,REFSPEC1: night12/night12.full233.ms.fits -> night12/night12.full233.ms.fits
night12/night12.233.ms.fits updated
night12/night12.234.ms.fits
night12/night12.234.ms.fits,REFSPEC1: night12/night12.full234.ms.fits -> night12/night12.full234.ms.fits
night12/night12.234.ms

night12/night12.c221.ms.fits: ap = 1, w1 =  4347.44, w2 = 6062.805, dw = 1.009632, nw = 1700
night12/night12.222.ms.fits: REFSPEC1 = 'night12/night12.full222.ms.fits 1.'
night12/night12.c222.ms.fits: ap = 1, w1 =  4346.99, w2 =  6062.85, dw = 1.009923, nw = 1700
night12/night12.225.ms.fits: REFSPEC1 = 'night12/night12.full225.ms.fits 1.'
night12/night12.c225.ms.fits: ap = 1, w1 = 4347.454, w2 = 6062.803, dw = 1.009623, nw = 1700
night12/night12.226.ms.fits: REFSPEC1 = 'night12/night12.full226.ms.fits 1.'
night12/night12.c226.ms.fits: ap = 1, w1 = 4347.189, w2 = 6062.824, dw = 1.009791, nw = 1700
night12/night12.229.ms.fits: REFSPEC1 = 'night12/night12.full229.ms.fits 1.'
night12/night12.c229.ms.fits: ap = 1, w1 = 4346.966, w2 = 6062.851, dw = 1.009938, nw = 1700
night12/night12.230.ms.fits: REFSPEC1 = 'night12/night12.full230.ms.fits 1.'
night12/night12.c230.ms.fits: ap = 1, w1 = 4346.618, w2 = 6062.876, dw = 1.010158, nw = 1700
night12/night12.233.ms.fits: REFSPEC1 = 'night12/night12.

...

Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t156.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t157.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t158.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t159.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t160.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t162.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t163.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t164.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t165.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t166.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t167.ms already exists
Sep 10 17:07: EXTRACT - Output spectrum night12/night12.155t169.ms already exists
Sep 10 17:07: EX

Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t151.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t152.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t153.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t154.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t156.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t157.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t158.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t159.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t160.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t162.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t163.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.168t164.ms already exists
Sep 10 17:08: EX

Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t146.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t147.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t149.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t150.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t151.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t152.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t153.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t154.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t156.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t157.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t158.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.183t159.ms already exists
Sep 10 17:08: EX

Sep 10 17:08: EXTRACT - Output spectrum night12/night12.190t202.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t143.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t144.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t145.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t146.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t147.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t149.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t150.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t151.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t152.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t153.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.196t154.ms already exists
Sep 10 17:08: EX

Sep 10 17:08: EXTRACT - Output spectrum night12/night12.203t198.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.203t199.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.203t200.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.203t201.ms already exists
Sep 10 17:08: EXTRACT - Output spectrum night12/night12.203t202.ms already exists
night12/night12.c143.ms.fits,CRPIX1: 1. -> 1.862744
night12/night12.c143.ms.fits updated
Corrected night12/night12.c143.ms.fits
night12/night12.c144.ms.fits,CRPIX1: 1. -> 1.887193
night12/night12.c144.ms.fits updated
Corrected night12/night12.c144.ms.fits
night12/night12.c145.ms.fits,CRPIX1: 1. -> 1.905031
night12/night12.c145.ms.fits updated
Corrected night12/night12.c145.ms.fits
night12/night12.c146.ms.fits,CRPIX1: 1. -> 1.922703
night12/night12.c146.ms.fits updated
Corrected night12/night12.c146.ms.fits
night12/night12.c147.ms.fits,CRPIX1: 1. -> 1.939918
night12/night12.c147.ms.fi

In [18]:
iraf.pwd()

/media/gregory/Genesis/Modspec/Modspec_Calibration


In [27]:


xe_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}xe.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
ne_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}ne.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
ar_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}ar.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
full_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}full.txt".format(Calib_Night)),
                          format="ascii.no_header", names=("wv", "int"))
second_full = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "night{0:d}".format(Calib_Night), 
                                      "night{0:d}.cfull234.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))

xe_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "xenon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=23, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
ne_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "neon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=25, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
ar_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "argon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=42, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
full_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "all.feat"),
                       format="ascii.fixed_width", header_start=2, data_end=84, guess=False, col_starts=(2, 11, 22, 33),
                      col_ends=(10, 21, 32, 43))
full2_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "night12", "full2.feat"),
                       format="ascii.fixed_width", header_start=2, data_end=73, guess=False, col_starts=(2, 11, 22, 33),
                      col_ends=(10, 21, 32, 43))

full_xefeat = full_feat[np.array([v in xe_feat["User"] for v in full_feat["User"]])]
full_nefeat = full_feat[np.array([v in ne_feat["User"] for v in full_feat["User"]])]
full_arfeat = full_feat[np.array([v in ar_feat["User"] for v in full_feat["User"]])]

full2_xefeat = full2_feat[np.array([v in xe_feat["User"] for v in full2_feat["User"]])]
full2_nefeat = full2_feat[np.array([v in ne_feat["User"] for v in full2_feat["User"]])]
full2_arfeat = full2_feat[np.array([v in ar_feat["User"] for v in full2_feat["User"]])]


pixels = np.arange(1700)+1

In [11]:
allfeatures = full_feat
init_model = models.Linear1D(slope=-1, intercept=6063)
fitter = fitting.LevMarLSQFitter()
disp = fitter(init_model, allfeatures["Pixel"], allfeatures["User"])

linear = disp(pixels)
xe_nonlinear = xe_solution["wv"] - linear
ne_nonlinear = ne_solution["wv"] - linear
ar_nonlinear = ar_solution["wv"] - linear

xefeat_nonlinear = full_xefeat["User"] - disp(full_xefeat["Pixel"])
nefeat_nonlinear = full_nefeat["User"] - disp(full_nefeat["Pixel"])
arfeat_nonlinear = full_arfeat["User"] - disp(full_arfeat["Pixel"])

plt.plot(pixels, xe_nonlinear, color=xc, marker=".", label="Xenon")
plt.plot(pixels, ne_nonlinear, color=nc, marker=".", label="Neon")
plt.plot(pixels, ar_nonlinear, color=ac, marker=".", label="Argon")
plt.legend(loc="upper left")
plt.plot(xe_feat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="")
plt.plot(ne_feat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="")
plt.plot(ar_feat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [21]:
full_nonlinear = full_solution["wv"] - linear

xefeat_nonlinear = full_xefeat["User"] - disp(full_xefeat["Pixel"])
nefeat_nonlinear = full_nefeat["User"] - disp(full_nefeat["Pixel"])
arfeat_nonlinear = full_arfeat["User"] - disp(full_arfeat["Pixel"])

plt.plot(pixels, full_nonlinear, color=fc, marker=".", label="Joint")
plt.plot(full_xefeat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full_nefeat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="", label="Neon")
plt.plot(full_arfeat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper left")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [35]:
full2_nonlinear = second_full["wv"] - linear

xefeat_nonlinear = full2_xefeat["User"] - disp(full2_xefeat["Pixel"])
nefeat_nonlinear = full2_nefeat["User"] - disp(full2_nefeat["Pixel"])
arfeat_nonlinear = full2_arfeat["User"] - disp(full2_arfeat["Pixel"])

plt.plot(pixels, full_nonlinear, color=fc, marker=".", label="Joint")
plt.plot(full2_xefeat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full2_nefeat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="", label="Neon")
plt.plot(full2_arfeat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="", label="Argon")
#plt.legend(loc="upper left")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [37]:
xefeat_residual = -full_xefeat["Residual"]
nefeat_residual = -full_nefeat["Residual"]
arfeat_residual = -full_arfeat["Residual"]

plt.plot(full_xefeat["Pixel"], xefeat_residual, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full_nefeat["Pixel"], nefeat_residual, color=nc, marker="o", ls="", label="Neon")
plt.plot(full_arfeat["Pixel"], arfeat_residual, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper right")
plt.plot([pixels[0], pixels[-1]], [0, 0], color=fc, ls="--")
plt.xlabel("Pixel")
plt.ylabel("Residual (User - Fit)")

In [39]:
xefeat_residual = -full2_xefeat["Residual"]
nefeat_residual = -full2_nefeat["Residual"]
arfeat_residual = -full2_arfeat["Residual"]

plt.plot(full2_xefeat["Pixel"], xefeat_residual, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full2_nefeat["Pixel"], nefeat_residual, color=nc, marker="o", ls="", label="Neon")
plt.plot(full2_arfeat["Pixel"], arfeat_residual, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper right")
plt.plot([pixels[0], pixels[-1]], [0, 0], color=fc, ls="--")
plt.xlabel("Pixel")
plt.ylabel("Residual (User - Fit)")
plt.title("Wavelength Solution 234")

In [36]:
soldiff = second_full["wv"] - full_solution["wv"]

plt.plot(pixels, soldiffm color=fc, ls=".")

common_xe = join(full_xefit, full2_xefit, )



SyntaxError: invalid syntax (<ipython-input-36-e9d790ce7f22>, line 3)

plt.show()

In [69]:
matplotlib.interactive(True)

In [14]:
plt.show()

KeyboardInterrupt: 

In [44]:
iraf.reidentify(os.path.join("calib_test", "arspec.fits"), "@Night12_Subtracted_Arcs.txt", intera="no")

In [79]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, "Night12_Arcs.txt"),"r") as arcs:
    arcfiles = arcs.readlines()
    slopes = np.zeros(shape=len(arcfiles))
    intercepts = np.zeros(shape=len(arcfiles))
    times = np.zeros(shape=len(arcfiles))
    airmasses = np.zeros(shape=len(arcfiles))
    for i, arcfile in enumerate(arcfiles):
        base, ext = os.path.splitext(arcfile[:-1])
        dbfile = "".join([os.path.join(IMAGE_PATH, CALIB_FOLDER, "database", "id"), base])
        # First look for the end of the data.
        with open(dbfile, "r") as database:
            fullfile = database.read()
        lastentry = fullfile[fullfile.rindex("begin"):]
        features_index = lastentry.index("features")
        tablelength = int(lastentry[features_index+8:features_index+lastentry[features_index:].index("\n")])
        feat_table = Table.read(lastentry, format="ascii.fixed_width", names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"),
                               data_start=6, data_end=6+tablelength, col_starts=(0, 15, 26, 37, 43, 45), 
                                col_ends=(14, 25, 36, 40, 44, 46))
        joined_feats = join(feat_table, full2_arfeat, join_type="inner", table_names=("STD", "ARC"), keys="User")
        ideal_fit = models.Linear1D(slope=1, intercept=0)
        fitter = fitting.LevMarLSQFitter()
        actual_fit = fitter(ideal_fit, joined_feats["Pixel_STD"], joined_feats["Pixel_ARC"])
        slopes[i] = actual_fit.slope.value
        intercepts[i] = actual_fit.intercept.value
        
        # Now let's get the time of observation and the airmass
        fitsfile = os.path.join(IMAGE_PATH, CALIB_FOLDER, arcfile[:-1])
        hdulist = fits.open(fitsfile)
        times[i] = hdulist[0].header["JD"]
        airmasses[i] = hdulist[0].header["AIRMASS"]

In [94]:
timediff = (times - times[0])*24
plt.plot(timediff, slopes, 'k+')
plt.plot([timediff[0], timediff[-1]], [1, 1], 'k--')
plt.xlabel("Time since first exposure")
plt.ylabel("Slope")
plt.title("Dispersion difference")

In [95]:
plt.plot(airmasses-1, slopes, 'k+')
plt.plot([0, 1], [1, 1], 'k--')
plt.xlabel("Airmass")
plt.ylabel("Slope")
plt.title("Dispersion difference")

In [14]:
# List of extracted spectra:
extracted_arspec_standards = "Night12_Standards_Ar.txt"
extracted_nespec_standards = "Night12_Standards_Ne.txt"
extracted_xespec_standards = "Night12_Standards_Xe.txt"
# Extracted spectra take the form of night{d}.{a}{n}.ms.fit
# d is the day of the observation. So between 1-12.
# a is the type of arc. So either ne, xe, or ar.
# n is the exposure of the object that was used for the trace.

In [22]:
# Let's subtract the continuum from the targets.
# Put them in the form night{d}.s{a}{n}.fit
iraf.splot.function = "chebyshev"
iraf.splot.order = 20
iraf.splot.low_reject = 0
iraf.splot.high_reject = 2
iraf.splot("@" + extracted_arspec_standards)
iraf.splot("@" + extracted_xespec_standards)
iraf.splot("@" + extracted_nespec_standards)

/=normalize, -=subtract, f=fit, c=clean, n=nop, q=quit/=normalize, -=subtract, f=fit, c=clean, n=nop, q=quit/=normalize, -=subtract, f=fit, c=clean, n=nop, q=quitwindow:again:window:again:window:again:window:window:again:window:window:again:window:







1. INTERACTIVE CURVE FITTING CURSOR OPTIONS

?	Print options
a	Add point to constrain fit
c	Print the coordinates and fit of point nearest the cursor
d	Delete data point nearest the cursor
f	Fit the data and redraw or overplot
g	Redefine graph keys.  Any of the following data types may be along
	either axis.
	    x  Independent variable	y  Dependent variable
	    f  Fitted value		r  Residual (y - f)
	    d  Ratio (y / f)		n  Nonlinear part of y
h-l	Graph keys.  Defaults are h=(x,y), i=(y,x), j=(x,r), k=(x,d), l=(x,n)
o	Overplot the next graph
q	Exit the interactive curve fitting.  Carriage return will also exit.
r	Redraw graph
s	Set sample range with the cursor
t	Initialize the sample range to all points
v	Change the weight of the point nearest the cursor
u	Undelete the deleted point nearest the cursor
w	Set the graph window.  For help type 'w' followed by '?'.
x	Change the x value of the point nearest the cursor
y	Change the y value of the point nearest the cursor
z	Delete sample regi

In [11]:
subtracted_arspec_standards = "Night{0:d}_Standards_Ar_Subtracted.txt".format(Calib_Night)
subtracted_nespec_standards = "Night{0:d}_Standards_Ne_Subtracted.txt".format(Calib_Night)
subtracted_xespec_standards = "Night{0:d}_Standards_Xe_Subtracted.txt".format(Calib_Night)

In [14]:
# Now what I want to do is wavelength-calibrate each of these spectra.
ar_ref = os.path.join("calib_test", "arspec.fits")
ne_ref = os.path.join("calib_test", "nespec.fits")
xe_ref = os.path.join("calib_test", "xespec.fits")
iraf.reidentify(ar_ref, "@" + subtracted_arspec_standards, coordlist="Calibrations/Argon_linelist.dat")
iraf.reidentify(ne_ref, "@" + subtracted_nespec_standards, coordlist="Calibrations/Neon_linelist.dat")
iraf.reidentify(xe_ref, "@" + subtracted_xespec_standards, coordlist="Calibrations/Xenon_linelist.dat")

In [7]:
fullspec_standards = "Night{0:d}_Standards_Fullspec.txt".format(Calib_Night)

In [40]:
# Now what I want to do is make one of the spectra designated as fullspec
# Extracted spectra take the form of night{d}.full{n}.fit
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, subtracted_arspec_standards)) as arimg, \
    open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullimg:
    for src, dst in zip(arimg, fullimg):
        shutil.copy(src[:-1], dst[:-1])

In [8]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspecs:
    for fullimg in fullspecs:
        # These are the image names
        arimg = fullimg.replace("full", "sar")
        neimg = fullimg.replace("full", "sne")
        xeimg = fullimg.replace("full", "sxe")
        # Now get the database names.
        ardb, ext = os.path.splitext(os.path.join("database", "id"+arimg))
        nedb, ext = os.path.splitext(os.path.join("database", "id"+neimg))
        xedb, ext = os.path.splitext(os.path.join("database", "id"+xeimg))
        fulldb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
        # First read in the argon entry.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, ardb)) as ardata:
            ar_fullfile = ardata.read()
        ar_features = ar_fullfile[ar_fullfile.rindex("begin"):]
        ar_length_line_start = ar_features.index("features")
        ar_length_line_end = ar_features.index("\n", ar_length_line_start)
        ar_numlines = int(ar_features[ar_length_line_start:ar_length_line_end].split("\t")[1])
        ar_table_start = ar_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ar_table_end = ar_features.index("function")-2
        ar_table = ar_features[ar_table_start:ar_table_end].split("\n")
        ar_feat_table = Table.read(ar_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                   col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        # Now read in the neon and xenon entries
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, nedb)) as nedata:
            ne_fullfile = nedata.read()
        ne_features = ne_fullfile[ne_fullfile.rindex("begin"):]
        ne_length_line_start = ne_features.index("features")
        ne_length_line_end = ne_features.index("\n", ne_length_line_start)
        ne_numlines = int(ne_features[ne_length_line_start:ne_length_line_end].split("\t")[1])
        ne_table_start = ne_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ne_table_end = ne_features.index("function")-2
        ne_table = ne_features[ne_table_start:ne_table_end].split("\n")
        ne_feat_table = Table.read(ne_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                   col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, xedb)) as xedata:
            xe_fullfile = xedata.read()
        xe_features = xe_fullfile[xe_fullfile.rindex("begin"):]
        xe_length_line_start = xe_features.index("features")
        xe_length_line_end = xe_features.index("\n", xe_length_line_start)
        xe_numlines = int(xe_features[xe_length_line_start:xe_length_line_end].split("\t")[1])
        xe_table_start = xe_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        xe_table_end = xe_features.index("function")-2
        xe_table = xe_features[xe_table_start:xe_table_end].split("\n")
        xe_feat_table = Table.read(xe_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                   col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        
        # Now combine the features.
        full_features = ar_table + ne_table + xe_table
        full_numlines = ar_numlines+ne_numlines+xe_numlines
        full_feat_table = Table.read(full_features, format="ascii.fixed_width_no_header", 
                                names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        full_feat_table["Count"] = np.arange(len(full_feat_table))
        full_feat_table.sort("Pixel")
        sorted_table = [full_features[i] for i in full_feat_table["Count"]]
        new_feature_table = "\n".join(sorted_table)

        # Now let's piece together the new file. First make the time comment.
        a = datetime.now()
        comment_line = "# " + a.strftime("%a %H:%M:%S %d-%b-%Y") + "\n"
        # Then make the header:
        ar_head_start = 0
        # Note that this includes the leading tab character in the header, not as part of the "feature" line.
        ar_head_end = ar_length_line_start
        ar_header = ar_features[ar_head_start:ar_head_end]
        full_header = ar_header.replace("sar", "full")
        # Now the feature line will be added on.
        full_feature_line = "features\t{0:d}\n".format(full_numlines)
        # Lastly we want the footer, which doesn't actually contain any useful information, but we will include.
        footer = ar_features[ar_table_end:]
        # Now add them all together!
        fullfile = comment_line + full_header + full_feature_line + new_feature_table + footer
        
        # Write the result to a file.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fulldb), 'a') as fulldata:
            fulldata.write(fullfile)

NameError: name 'fullspec_standards' is not defined

In [27]:
matplotlib.interactive(False)
# Now let's go through the images and generate plots of the residuals.
# I'll want to save them in a subdirectory in the plots folder.
calib_residual_path = os.path.join(os.environ["THESIS"], "plots", "calib_residuals")
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspecs:
    for fullimg in fullspecs:
        # These are the image names
        arimg = fullimg.replace("full", "sar")
        neimg = fullimg.replace("full", "sne")
        xeimg = fullimg.replace("full", "sxe")
        # Now get the database names.
        ardb, ext = os.path.splitext(os.path.join("database", "id"+arimg))
        nedb, ext = os.path.splitext(os.path.join("database", "id"+neimg))
        xedb, ext = os.path.splitext(os.path.join("database", "id"+xeimg))
        # Now the main feature file.
        featfile = os.path.splitext(os.path.splitext(fullimg)[0])[0] + ".feat"
                # First read in the argon entry.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, ardb)) as ardata:
            ar_fullfile = ardata.read()
        ar_features = ar_fullfile[ar_fullfile.rindex("begin"):]
        ar_length_line_start = ar_features.index("features")
        ar_length_line_end = ar_features.index("\n", ar_length_line_start)
        ar_numlines = int(ar_features[ar_length_line_start:ar_length_line_end].split("\t")[1])
        ar_table_start = ar_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ar_table_end = ar_features.index("function")-2
        ar_table = ar_features[ar_table_start:ar_table_end].split("\n")
        ar_feat_table = Table.read(ar_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        # Now read in the neon and xenon entries
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, nedb)) as nedata:
            ne_fullfile = nedata.read()
        ne_features = ne_fullfile[ne_fullfile.rindex("begin"):]
        ne_length_line_start = ne_features.index("features")
        ne_length_line_end = ne_features.index("\n", ne_length_line_start)
        ne_numlines = int(ne_features[ne_length_line_start:ne_length_line_end].split("\t")[1])
        ne_table_start = ne_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ne_table_end = ne_features.index("function")-2
        ne_table = ne_features[ne_table_start:ne_table_end].split("\n")
        ne_feat_table = Table.read(ne_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, xedb)) as xedata:
            xe_fullfile = xedata.read()
        xe_features = xe_fullfile[xe_fullfile.rindex("begin"):]
        xe_length_line_start = xe_features.index("features")
        xe_length_line_end = xe_features.index("\n", xe_length_line_start)
        xe_numlines = int(xe_features[xe_length_line_start:xe_length_line_end].split("\t")[1])
        xe_table_start = xe_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        xe_table_end = xe_features.index("function")-2
        xe_table = xe_features[xe_table_start:xe_table_end].split("\n")
        xe_feat_table = Table.read(xe_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        # Now get the full feature table
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, featfile)) as ffeat:
            full_feat = ffeat.read()
        feat_entry = full_feat[full_feat.rindex("Features identified"):]
        full_feat = Table.read(feat_entry, format="ascii.fixed_width", header_start=1, data_end=-1, guess=False, 
                               col_starts=(2, 11, 22, 33), col_ends=(10, 21, 32, 43))
        # Now break it up by element.
        ne_feats = join(full_feat, ne_feat_table[["User"]])
        xe_feats = join(full_feat, xe_feat_table[["User"]])
        ar_feats = join(full_feat, ar_feat_table[["User"]])                          
        
        # Now plot the residuals
        plt.plot(ne_feats["User"], -ne_feats["Residual"], color=nc, marker="o", ls="", label="Neon")
        plt.plot(xe_feats["User"], -xe_feats["Residual"], color=xc, marker="d", ls="", label="Xenon")
        plt.plot(ar_feats["User"], -ar_feats["Residual"], color=ac, marker="s", ls="", label="Argon")
        plt.xlabel("Wavelength (A)")
        plt.ylabel("Residual (User - Fit)")
        plt.title("Full Fit Residual for {0}".format(os.path.basename(fullimg)))
        plt.legend(loc="upper left")
        plt.savefig(os.path.join(calib_residual_path, os.path.splitext(os.path.basename(featfile))[0]+".png"))
        plt.close()

In [8]:
calibrated_standards = "Night12_Standards_Calib.txt"

In [42]:
# Now apply the wavelength calibration to the images.
# First associate each object spectrum with the arc spectrum.
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspec, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standards)) as stand:
        for ref, obj in zip(fullspec, stand):
            print obj[:-1]
            iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand:
    for cstan in calstand:
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, cstan[:-1]))
        except OSError:
            pass
iraf.dispcor("@"+extracted_standards, "@"+calibrated_standards, linearize=True)

night12/night12.076.ms.fits
add night12/night12.076.ms.fits,REFSPEC1 = night12/night12.full076.ms.fits
night12/night12.076.ms.fits updated
night12/night12.079.ms.fits
add night12/night12.079.ms.fits,REFSPEC1 = night12/night12.full079.ms.fits
night12/night12.079.ms.fits updated
night12/night12.080.ms.fits
add night12/night12.080.ms.fits,REFSPEC1 = night12/night12.full080.ms.fits
night12/night12.080.ms.fits updated
night12/night12.083.ms.fits
add night12/night12.083.ms.fits,REFSPEC1 = night12/night12.full083.ms.fits
night12/night12.083.ms.fits updated
night12/night12.084.ms.fits
add night12/night12.084.ms.fits,REFSPEC1 = night12/night12.full084.ms.fits
night12/night12.084.ms.fits updated
night12/night12.087.ms.fits
add night12/night12.087.ms.fits,REFSPEC1 = night12/night12.full087.ms.fits
night12/night12.087.ms.fits updated
night12/night12.088.ms.fits
add night12/night12.088.ms.fits,REFSPEC1 = night12/night12.full088.ms.fits
night12/night12.088.ms.fits updated
night12/night12.091.ms.fits

night12/night12.c091.ms.fit: ap = 1, w1 = 4346.863, w2 = 6062.853, dw =     1.01, nw = 1700
night12/night12.092.ms.fits: REFSPEC1 = 'night12/night12.full092.ms.fits 1.'
night12/night12.c092.ms.fit: ap = 1, w1 = 4346.607, w2 = 6062.877, dw = 1.010165, nw = 1700
night12/night12.095.ms.fits: REFSPEC1 = 'night12/night12.full095.ms.fits 1.'
night12/night12.c095.ms.fit: ap = 1, w1 = 4347.328, w2 = 6062.814, dw = 1.009704, nw = 1700
night12/night12.096.ms.fits: REFSPEC1 = 'night12/night12.full096.ms.fits 1.'
night12/night12.c096.ms.fit: ap = 1, w1 =   4347.3, w2 = 6062.816, dw = 1.009721, nw = 1700
night12/night12.099.ms.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.c099.ms.fit: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.100.ms.fits: REFSPEC1 = 'night12/night12.full100.ms.fits 1.'
night12/night12.c100.ms.fit: ap = 1, w1 = 4346.648, w2 = 6062.877, dw =  1.01014, nw = 1700
night12/night12.103.ms.fits: REFSPEC1 = 'night12/night12.full10

# Flexure

In [7]:
arc_files = "Night{0:d}_Arcs.txt".format(Calib_Night)
# Now let's assign wavelength calibrations to the arcs.
# First we have to extract arc spectra using a given trace. Let's pick 079.
extracted_arcs = "Night{0:d}_Arcs_Extracted.txt".format(Calib_Night)

standard_arcs = "Night12_Standards_Arcs.txt"
extracted_arcs = "Night12_Standards_Arcs_Extracted.txt"
calibrated_arcs = "Night12_Standards_Arcs_Calib.txt"
argon_specs = "Night12_Standards_Ar.txt"
calibrated_argon_spec = "Night12_Standards_Ar_Calib.txt"

In [10]:
iraf.apall("@"+arc_files, out="@"+extracted_arcs, 
           ref=os.path.join("night{0:d}", "night{0:d}.079.fit").format(Calib_Night), recen=False, trace=False, 
           back="none", intera=False)

Aug 25  9:39: EXTRACT - Output spectrum night12/night12.077t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.078t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.081t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.082t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.085t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.086t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.089t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.090t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.093t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.094t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.097t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/night12.098t079 already exists
Aug 25  9:39: EXTRACT - Output spectrum night12/nigh

In [ ]:
iraf.fxcor.pixcorr = True
iraf.fxcor.function = "gaussian"

iraf.continpars.c_inter = True
iraf.continpars.order = 20
iraf.continpars.low_rej = 0
iraf.continpars.high_rej = 2
iraf.continpars.nitera = 10
iraf.continpars.grow = 1

with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs) as exarc_file), \
        open(os.path.join(IMAGE_PATH, CALIB_FOLDER, argon_specs) as arspec):
    for arc, ar in zip(exarc_file, arspec):
        outputroot = os.path.splitext(os.path.splitext(arc[:-1])[0])[0]
        fxcor(arc[:-1], ar[:-1], output=outputroot)
    
        

In [24]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarc_file:
    exarc_files = exarc_file.readlines()
times = np.zeros(len(exarc_files))
airmasses = np.zeros(len(exarc_files))
shifts = np.zeros(len(exarc_files))
for i, exarc in enumerate(exarc_files):
    # Record the MJD time and airmass of observation.
    fitsfile = os.path.join(IMAGE_PATH, CALIB_FOLDER, exarc[:-1])
    hdulist = fits.open(fitsfile)
    times[i] = hdulist[0].header["JD"]
    airmasses[i] = hdulist[0].header["AIRMASS"]
    
    # Now get the pixel shift.
    shiftfile = os.path.splitext(fitsfile)[0] + ".txt"
    shift_table = Table.read(shiftfile, format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                             header_start=13, guess=False, 
                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                    'VOBS', 'VREL', 'VHELIO', 'VERR'])
    shifts[i] = shift_table["SHIFT"][-1]
    hdulist.close()
times = times - times[0]

In [29]:
plt.plot(times*24, shifts, 'ks')
plt.xlabel("Hours since first observation")
plt.ylabel("Pixel shift")
plt.title("Flexure on Night 12")

In [38]:
early_color = (146/255.0, 0, 0)
mid_color = (0, 0, 0)
kic_color = (36/255.0, 255/255.0, 36/255.0)
late_color = (182/255.0, 219/255.0, 255/255.0)

In [44]:
early_indices = np.where(times*24 <= 0.6)
late_indices = np.where(times*24 >= 7.46)
mid_indices = np.where(np.logical_and(times*24 > 0.6, times*24 < 1.83))
kic_indices = np.where(np.logical_and(times*24 > 1.83, times*24 < 7.46))

plt.figure()
plt.plot(times[early_indices]*24, shifts[early_indices], c=early_color, ls="", marker="s", label="Early")
plt.plot(times[mid_indices]*24, shifts[mid_indices], c=mid_color, ls="", marker="*", label="Mid")
plt.plot(times[kic_indices]*24, shifts[kic_indices], c=kic_color, ls="", marker="d", label="KIC")
plt.plot(times[late_indices]*24, shifts[late_indices], c=late_color, ls="", marker="o", label="Late")
plt.xlabel("Hours since first observation")
plt.ylabel("Pixel shift")
plt.title("Flexure on Night 12")
plt.legend(loc="lower left")

plt.figure()
plt.plot(airmasses[early_indices], shifts[early_indices], c=early_color, ls="", marker="s", label="Early")
plt.plot(airmasses[mid_indices], shifts[mid_indices], c=mid_color, ls="", marker="*", label="Mid")
plt.plot(airmasses[kic_indices], shifts[kic_indices], c=kic_color, ls="", marker="d", label="KIC")
plt.plot(airmasses[late_indices], shifts[late_indices], c=late_color, ls="", marker="o", label="Late")
plt.xlabel("Airmass")
plt.ylabel("Pixel shift")
plt.title("Flexure on Night 12")
plt.legend(loc="upper right")

In [92]:
# Extract each of the arcs with traces over all KIC objects.
kic_arcfile = "Night12_KIC_Objects_Arcs.txt"
kic_targets = "Night12_KIC_Objects.txt"
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcfile), 'r') as arcs:
    total_shifts = []
    for arc in arcs:
        arcnum = arc[-8:-5]
        output_traces = "Night12_KIC_Objects_Arc{arcnum}.txt".format(arcnum=arcnum)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_targets), 'r') as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'w') as newfile:
            for oldname in oldfile:
                # night12.186.fit -> night12.187t186.ms.fits
                kicnum = oldname[-8:-5]
                newname = oldname.replace(".fit", ".ms.fits").replace(kicnum, arcnum+"t"+kicnum)
                newfile.write(newname)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_targets), 'r') as kics, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as outputs:
                for kic, out in zip(kics, outputs):
                    iraf.apall(arc[:-1], out=out[:-1], ref=kic[:-1], recen=False, trace=False, back="none", intera=False)
# Cross-correlate each arc with each argon calibration with the same trace.
        fxcor_base = "Night12_KIC_Object_Arc{arcnum}_FXcor.txt".format(arcnum=arcnum)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base), 'w') as newfile:
            for oldname in oldfile:
                # night12.187t186.ms.fits -> night12.187t186
                newname = os.path.splitext(os.path.splitext(oldname)[0])[0]+"\n"
                newfile.write(newname)
        iraf.fxcor.continuum = "both"
        iraf.fxcor.filter = "both"
        iraf.fxcor.pixcorr = "yes"
        iraf.fxcor.function = "gaussian"
        iraf.fxcor.observatory = "kpno"
    
        iraf.continpars.c_inter = True
        iraf.continpars.order = 20
        iraf.continpars.low_rej = 0
        iraf.continpars.high_rej = 2
        iraf.continpars.nitera = 10
        iraf.continpars.grow = 1
    
        iraf.filtpars.f_type = "square"
        iraf.filtpars.cuton = 30
        iraf.filtpars.cutoff = 1000
    
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base)) as corfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces)) as arcfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, 
                               calibration_template.format(n, obj_types[1], arctypes[2].capitalize())), 'r') as tempfile:
            arcshifts = []
            for corbase, arc, template in zip(corfile, arcfile, tempfile):
                iraf.fxcor(arc[:-1], template[:-1], out=corbase[:-1], interact="no")
                               
            # Measure scatter in flexure
                shiftfile = corbase[:-1] + ".txt"
                shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile), 
                                         format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                         header_start=13, guess=False, 
                                         names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                                'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
                shift = shift_table["SHIFT"][-1]
                arcshifts.append(shift)
    
        total_shifts.append(arcshifts)

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t143.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t144.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t145.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t146.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t147.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t149.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t150.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t151.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t152.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t153.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t154.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.142t156.ms already exists
Sep 10 12:27: EX

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.148t199.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.148t200.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.148t201.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.148t202.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t143.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t144.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t145.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t146.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t147.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t149.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t150.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.155t151.ms already exists
Sep 10 12:27: EX

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t194.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t195.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t197.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t198.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t199.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t200.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t201.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.161t202.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.168t143.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.168t144.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.168t145.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.168t146.ms already exists
Sep 10 12:27: EX

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t189.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t191.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t192.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t193.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t194.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t195.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t197.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t198.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t199.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t200.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t201.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.175t202.ms already exists
Sep 10 12:27: EX

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t185.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t186.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t187.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t188.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t189.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t191.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t192.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t193.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t194.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t195.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t197.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.190t198.ms already exists
Sep 10 12:27: EX

Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t180.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t181.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t182.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t184.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t185.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t186.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t187.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t188.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t189.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t191.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t192.ms already exists
Sep 10 12:27: EXTRACT - Output spectrum night12/night12.203t193.ms already exists
Sep 10 12:27: EX

In [99]:
shifts = np.array(total_shifts)
uncertainty = np.mean(np.std(shifts, axis=1))
print(uncertainty)

0.00148892862377


# Spectral mismatch variation

In [7]:
nightno = 1
standard_file = calibrated_target_template.format(nightno, obj_types[0])

In [180]:
# Read in the Standard information
standard_info = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "Standard_SIMBAD.txt"), 
                            format="ascii.commented_header", header_start=0, data_start=4, data_end=-1, delimiter="|", 
                           fill_values=[("~", 0), ("", 0)], guess=False)
rv_lookup = dict(zip(standard_info["typed ident"], standard_info["radvel"]))
coord_lookup = dict(zip(standard_info["typed ident"], SkyCoord(standard_info["coord1 (ICRS,J2000/2000)"], 
                                                               unit=(u.hourangle, u.deg))))

In [175]:
calibrated_target_

('Standards', 'KIC_Objects')

In [178]:
# Check that all of the exposures are actually pointing at the objects they claim to be.
for n in obsnights:
    standards = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standards)) as stand_file:
        for stand in stand_file:
            objlabel = iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            ra = iraf.hedit(stand[:-1], "RA", ".", Stdout=1)[0].split("=")[1].strip()
            dec = iraf.hedit(stand[:-1], "DEC", ".", Stdout=1)[0].split("=")[1].strip()
            filecoord = SkyCoord(ra, dec, unit=(u.hourangle, u.deg))
            standard_coord = coord_lookup[objlabel]
            offset = filecoord.separation(standard_coord)
            if offset > 2*u.arcmin:
                print "{0} offset is: {1}'".format(stand[:-1], offset.to(u.arcmin))

night12/night12.c136.ms.fits offset is: 1609.98092894 arcmin'


In [179]:
# Insert the known velocity in all of the targets.
for n in obsnights:
    standards_list = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standards_list)) as standimages:
        for stand in standimages:
            objlabel = iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            try:
                rv = rv_lookup[objlabel]
            except KeyError:
                print "Could not find entry for {0} in file {1}".format(objlabel, stand[:-1])
            iraf.hedit(stand[:-1], "VHELIO", str(rv), add="yes", verify="no")
            print(stand[:-1], rv)

night1/night1.c072.ms.fits,VHELIO: -29 -> -29.39
night1/night1.c072.ms.fits updated
('night1/night1.c072.ms.fits', -29.390000000000001)
night1/night1.cd01.ms.fits,VHELIO: -8 -> -7.73
night1/night1.cd01.ms.fits updated
('night1/night1.cd01.ms.fits', -7.7300000000000004)
night1/night1.c077.ms.fits,VHELIO: -5 -> -4.79
night1/night1.c077.ms.fits updated
('night1/night1.c077.ms.fits', -4.79)
night1/night1.cd02.ms.fits,VHELIO: -45 -> -45.55
night1/night1.cd02.ms.fits updated
('night1/night1.cd02.ms.fits', -45.549999999999997)
night1/night1.c115.ms.fits,VHELIO: -33 -> -32.83
night1/night1.c115.ms.fits updated
('night1/night1.c115.ms.fits', -32.829999999999998)
night1/night1.c118.ms.fits,VHELIO: 5 -> 5.4
night1/night1.c118.ms.fits updated
('night1/night1.c118.ms.fits', 5.4000000000000004)
night1/night1.c119.ms.fits,VHELIO: -18 -> -17.95
night1/night1.c119.ms.fits updated
('night1/night1.c119.ms.fits', -17.949999999999999)
night1/night1.cd03.ms.fits,VHELIO: 20 -> 19.98
night1/night1.cd03.ms.fit

night4/night4.c116.ms.fits,VHELIO: 15 -> 15.12
night4/night4.c116.ms.fits updated
('night4/night4.c116.ms.fits', 15.119999999999999)
night4/night4.c119.ms.fits,VHELIO: -8 -> -7.73
night4/night4.c119.ms.fits updated
('night4/night4.c119.ms.fits', -7.7300000000000004)
night4/night4.c120.ms.fits,VHELIO: -47 -> -73.03
night4/night4.c120.ms.fits updated
('night4/night4.c120.ms.fits', -73.030000000000001)
night4/night4.c123.ms.fits,VHELIO: -60 -> -59.51
night4/night4.c123.ms.fits updated
('night4/night4.c123.ms.fits', -59.509999999999998)
night4/night4.c124.ms.fits,VHELIO: -45 -> -44.55
night4/night4.c124.ms.fits updated
('night4/night4.c124.ms.fits', -44.549999999999997)
night4/night4.c127.ms.fits,VHELIO: 20 -> 19.98
night4/night4.c127.ms.fits updated
('night4/night4.c127.ms.fits', 19.98)
night4/night4.c128.ms.fits,VHELIO: -94 -> -94.06
night4/night4.c128.ms.fits updated
('night4/night4.c128.ms.fits', -94.060000000000002)
night4/night4.c131.ms.fits,VHELIO: -5 -> -4.79
night4/night4.c131.ms.

('night5/night5.c150.ms.fits', 9.5299999999999994)
night5/night5.c151.ms.fits,VHELIO: -10 -> -10.22
night5/night5.c151.ms.fits updated
('night5/night5.c151.ms.fits', -10.220000000000001)
night5/night5.c206.ms.fits,VHELIO: -45 -> -45.55
night5/night5.c206.ms.fits updated
('night5/night5.c206.ms.fits', -45.549999999999997)
night5/night5.c207.ms.fits,VHELIO: -94 -> -94.06
night5/night5.c207.ms.fits updated
('night5/night5.c207.ms.fits', -94.060000000000002)
night5/night5.c210.ms.fits,VHELIO: -10 -> -10.22
night5/night5.c210.ms.fits updated
('night5/night5.c210.ms.fits', -10.220000000000001)
night5/night5.c211.ms.fits,VHELIO: -24 -> -24.5
night5/night5.c211.ms.fits updated
('night5/night5.c211.ms.fits', -24.5)
night5/night5.c214.ms.fits,VHELIO: -43 -> -42.93
night5/night5.c214.ms.fits updated
('night5/night5.c214.ms.fits', -42.93)
night5/night5.c215.ms.fits,VHELIO: -42 -> -42.42
night5/night5.c215.ms.fits updated
('night5/night5.c215.ms.fits', -42.420000000000002)
night5/night5.c218.ms.fit

night6/night6.c258.ms.fits,VHELIO: -33 -> -32.83
night6/night6.c258.ms.fits updated
('night6/night6.c258.ms.fits', -32.829999999999998)
night6/night6.c261.ms.fits,VHELIO: -45 -> -45.55
night6/night6.c261.ms.fits updated
('night6/night6.c261.ms.fits', -45.549999999999997)
night6/night6.c262.ms.fits,VHELIO: -41 -> -41.08
night6/night6.c262.ms.fits updated
('night6/night6.c262.ms.fits', -41.079999999999998)
night6/night6.c265.ms.fits,VHELIO: -121 -> -121.19
night6/night6.c265.ms.fits updated
('night6/night6.c265.ms.fits', -121.19)
night6/night6.c266.ms.fits,VHELIO: -47 -> -46.66
night6/night6.c266.ms.fits updated
('night6/night6.c266.ms.fits', -46.659999999999997)
night6/night6.c269.ms.fits,VHELIO: -6 -> -6.0
night6/night6.c269.ms.fits updated
('night6/night6.c269.ms.fits', -6.0)
night6/night6.c270.ms.fits,VHELIO: -140 -> -139.69
night6/night6.c270.ms.fits updated
('night6/night6.c270.ms.fits', -139.69)
night6/night6.c273.ms.fits,VHELIO: -20 -> -20.45
night6/night6.c273.ms.fits updated
('

('night9/night9.c202.ms.fits', -32.829999999999998)
night9/night9.c205.ms.fits,VHELIO: -45 -> -45.55
night9/night9.c205.ms.fits updated
('night9/night9.c205.ms.fits', -45.549999999999997)
night9/night9.c206.ms.fits,VHELIO: -41 -> -41.08
night9/night9.c206.ms.fits updated
('night9/night9.c206.ms.fits', -41.079999999999998)
night9/night9.c209.ms.fits,VHELIO: -121 -> -121.19
night9/night9.c209.ms.fits updated
('night9/night9.c209.ms.fits', -121.19)
night9/night9.c210.ms.fits,VHELIO: -47 -> -46.66
night9/night9.c210.ms.fits updated
('night9/night9.c210.ms.fits', -46.659999999999997)
night9/night9.c213.ms.fits,VHELIO: -6 -> -6.0
night9/night9.c213.ms.fits updated
('night9/night9.c213.ms.fits', -6.0)
night9/night9.c214.ms.fits,VHELIO: -140 -> -139.69
night9/night9.c214.ms.fits updated
('night9/night9.c214.ms.fits', -139.69)
night9/night9.c217.ms.fits,VHELIO: -20 -> -20.45
night9/night9.c217.ms.fits updated
('night9/night9.c217.ms.fits', -20.449999999999999)
night9/night9.c218.ms.fits,VHELIO:

('night11/night11.c081.ms.fits', -9.9499999999999993)
night11/night11.c084.ms.fits,VHELIO: -19 -> -18.79
night11/night11.c084.ms.fits updated
('night11/night11.c084.ms.fits', -18.789999999999999)
night11/night11.c085.ms.fits,VHELIO: -5 -> -4.91
night11/night11.c085.ms.fits updated
('night11/night11.c085.ms.fits', -4.9100000000000001)
night11/night11.c088.ms.fits,VHELIO: 5 -> 4.87
night11/night11.c088.ms.fits updated
('night11/night11.c088.ms.fits', 4.8700000000000001)
night11/night11.c089.ms.fits,VHELIO: -3 -> -2.36
night11/night11.c089.ms.fits updated
('night11/night11.c089.ms.fits', -2.3599999999999999)
night11/night11.c092.ms.fits,VHELIO: -35 -> -35.05
night11/night11.c092.ms.fits updated
('night11/night11.c092.ms.fits', -35.049999999999997)
night11/night11.c093.ms.fits,VHELIO: -4 -> -3.6
night11/night11.c093.ms.fits updated
('night11/night11.c093.ms.fits', -3.6000000000000001)
night11/night11.c096.ms.fits,VHELIO: -39 -> -39.32
night11/night11.c096.ms.fits updated
('night11/night11.

('night12/night12.c111.ms.fits', -9.9499999999999993)
night12/night12.c112.ms.fits,VHELIO: -16 -> -16.12
night12/night12.c112.ms.fits updated
('night12/night12.c112.ms.fits', -16.120000000000001)
night12/night12.c115.ms.fits,VHELIO: 31 -> 29.39
night12/night12.c115.ms.fits updated
('night12/night12.c115.ms.fits', 29.390000000000001)
night12/night12.c116.ms.fits,VHELIO: -45 -> -44.57
night12/night12.c116.ms.fits updated
('night12/night12.c116.ms.fits', -44.57)
night12/night12.c119.ms.fits,VHELIO: -15 -> -14.69
night12/night12.c119.ms.fits updated
('night12/night12.c119.ms.fits', -14.69)
night12/night12.c120.ms.fits,VHELIO: -30 -> -30.62
night12/night12.c120.ms.fits updated
('night12/night12.c120.ms.fits', -30.620000000000001)
night12/night12.c123.ms.fits,VHELIO: -42 -> -42.04
night12/night12.c123.ms.fits updated
('night12/night12.c123.ms.fits', -42.039999999999999)
night12/night12.c124.ms.fits,VHELIO: -33 -> -32.87
night12/night12.c124.ms.fits updated
('night12/night12.c124.ms.fits', -3

('night13/night13.c129.ms.fits', -29.390000000000001)
night13/night13.c130.ms.fits,VHELIO: 11 -> 11.38
night13/night13.c130.ms.fits updated
('night13/night13.c130.ms.fits', 11.380000000000001)
night13/night13.c133.ms.fits,VHELIO: -70 -> -72.42
night13/night13.c133.ms.fits updated
('night13/night13.c133.ms.fits', -72.420000000000002)
night13/night13.c134.ms.fits,VHELIO: -94 -> -94.06
night13/night13.c134.ms.fits updated
('night13/night13.c134.ms.fits', -94.060000000000002)
night13/night13.c200.ms.fits,VHELIO: 20 -> 19.98
night13/night13.c200.ms.fits updated
('night13/night13.c200.ms.fits', 19.98)
night13/night13.c201.ms.fits,VHELIO: -66 -> -66.1
night13/night13.c201.ms.fits updated
('night13/night13.c201.ms.fits', -66.099999999999994)
night13/night13.c204.ms.fits,VHELIO: -90 -> -86.99
night13/night13.c204.ms.fits updated
('night13/night13.c204.ms.fits', -86.989999999999995)
night13/night13.c205.ms.fits,VHELIO: -10 -> -10.22
night13/night13.c205.ms.fits updated
('night13/night13.c205.ms.

('night14/night14.c219.ms.fits', -0.45000000000000001)
night14/night14.c220.ms.fits,VHELIO: 24 -> 23.58
night14/night14.c220.ms.fits updated
('night14/night14.c220.ms.fits', 23.579999999999998)
night14/night14.c223.ms.fits,VHELIO: -60 -> -60.17
night14/night14.c223.ms.fits updated
('night14/night14.c223.ms.fits', -60.170000000000002)
night14/night14.c224.ms.fits,VHELIO: 8 -> 8.25
night14/night14.c224.ms.fits updated
('night14/night14.c224.ms.fits', 8.25)


'Night4_KIC_Objects_Calib.txt'

In [183]:
iraf.fxcor.high_rej = 0
iraf.fxcor.low_rej = 2
iraf.continpars.order = 15
iraf.fxcor.pixcor = "no"
iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"

# This function will first pick out template standards, and then cross-correlate the standard of all the nights.
for tempnight in obsnights:
    template_list = calibrated_target_template.format(tempnight, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_list)) as template_stands:
        templates = template_stands.readlines()
    for template in templates:
        # Now begin going through targets.
        for targnight in obsnights:
            template_cor_file = target_cor_template.format(targnight, obj_types[0], compact_standard(template.upper()))
            target_list = calibrated_target_template.format(targnight, obj_types[0])
            # Now populate template_cor_file
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file), "w") as newfile:
                    for oldname in oldfile:
                        newname = oldname.replace(".ms.fits", compact_standard(template))
                        newfile.write(newname)
            # Now cross-correlate.
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as targets, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as outputs:
                    for targ, out in zip(targets, outputs):
                        if targ != template:
                            iraf.fxcor(targ[:-1], template[:-1], out=out[:-1], interact="no")
                            print "Cross-correlating {0}".format(out[:-1])
                        else:
                            print "Skipping {0}".format(out[:-1])

Skipping night1/night1.c072n1c072
Cross-correlating night1/night1.cd01n1c072
Cross-correlating night1/night1.c077n1c072
Cross-correlating night1/night1.cd02n1c072
Cross-correlating night1/night1.c115n1c072
Cross-correlating night1/night1.c118n1c072
Cross-correlating night1/night1.c119n1c072
Cross-correlating night1/night1.cd03n1c072
Cross-correlating night1/night1.c124n1c072
Cross-correlating night1/night1.cd04n1c072
Cross-correlating night1/night1.cd05n1c072
Cross-correlating night3/night3.c081n1c072
Cross-correlating night3/night3.c083n1c072
Cross-correlating night3/night3.c086n1c072
Cross-correlating night3/night3.c087n1c072
Cross-correlating night3/night3.c090n1c072
Cross-correlating night3/night3.cd01n1c072
Cross-correlating night3/night3.c095n1c072
Cross-correlating night3/night3.c096n1c072
Cross-correlating night3/night3.c100n1c072
Cross-correlating night3/night3.c102n1c072
Cross-correlating night3/night3.c103n1c072
Cross-correlating night3/night3.c106n1c072
Cross-correlating ni

Cross-correlating night6/night6.c269n1c072
Cross-correlating night6/night6.c270n1c072
Cross-correlating night6/night6.c273n1c072
Cross-correlating night6/night6.c274n1c072
Cross-correlating night6/night6.c277n1c072
Cross-correlating night6/night6.c278n1c072
Cross-correlating night6/night6.c281n1c072
Cross-correlating night6/night6.c282n1c072
Cross-correlating night6/night6.c285n1c072
Cross-correlating night6/night6.c286n1c072
Cross-correlating night8/night8.c072n1c072
Cross-correlating night8/night8.c147n1c072
Cross-correlating night8/night8.c148n1c072
Cross-correlating night8/night8.c151n1c072
Cross-correlating night8/night8.c152n1c072
Cross-correlating night8/night8.c155n1c072
Cross-correlating night8/night8.c156n1c072
Cross-correlating night8/night8.c159n1c072
Cross-correlating night8/night8.c160n1c072
Cross-correlating night8/night8.c163n1c072
Cross-correlating night8/night8.c164n1c072
Cross-correlating night8/night8.c167n1c072
Cross-correlating night8/night8.c168n1c072
Cross-corre

Cross-correlating night12/night12.c128n1c072
Cross-correlating night12/night12.c131n1c072
Cross-correlating night12/night12.c132n1c072
Cross-correlating night12/night12.c135n1c072
Cross-correlating night12/night12.c136n1c072
Cross-correlating night12/night12.c139n1c072
Cross-correlating night12/night12.c140n1c072
Cross-correlating night12/night12.c205n1c072
Cross-correlating night12/night12.c206n1c072
Cross-correlating night12/night12.c209n1c072
Cross-correlating night12/night12.c210n1c072
Cross-correlating night12/night12.c213n1c072
Cross-correlating night12/night12.c214n1c072
Cross-correlating night12/night12.c217n1c072
Cross-correlating night12/night12.c218n1c072
Cross-correlating night12/night12.c221n1c072
Cross-correlating night12/night12.c222n1c072
Cross-correlating night12/night12.c225n1c072
Cross-correlating night12/night12.c226n1c072
Cross-correlating night12/night12.c229n1c072
Cross-correlating night12/night12.c230n1c072
Cross-correlating night12/night12.c233n1c072
Cross-corr

Cross-correlating night4/night4.c192n1cd01
Cross-correlating night4/night4.c193n1cd01
Cross-correlating night4/night4.c196n1cd01
Cross-correlating night4/night4.c197n1cd01
Cross-correlating night4/night4.c200n1cd01
Cross-correlating night4/night4.c201n1cd01
Cross-correlating night4/night4.c204n1cd01
Cross-correlating night4/night4.cd02n1cd01
Cross-correlating night4/night4.c209n1cd01
Cross-correlating night4/night4.c210n1cd01
Cross-correlating night4/night4.c213n1cd01
Cross-correlating night4/night4.c214n1cd01
Cross-correlating night4/night4.c217n1cd01
Cross-correlating night4/night4.c218n1cd01
Cross-correlating night4/night4.c221n1cd01
Cross-correlating night5/night5.c077n1cd01
Cross-correlating night5/night5.c078n1cd01
Cross-correlating night5/night5.c081n1cd01
Cross-correlating night5/night5.cd01n1cd01
Cross-correlating night5/night5.c086n1cd01
Cross-correlating night5/night5.c087n1cd01
Cross-correlating night5/night5.c090n1cd01
Cross-correlating night5/night5.c091n1cd01
Cross-corre

Cross-correlating night10/night10.c083n1cd01
Cross-correlating night10/night10.c086n1cd01
Cross-correlating night10/night10.c089n1cd01
Cross-correlating night10/night10.c090n1cd01
Cross-correlating night10/night10.c093n1cd01
Cross-correlating night10/night10.c094n1cd01
Cross-correlating night10/night10.c097n1cd01
Cross-correlating night10/night10.c098n1cd01
Cross-correlating night10/night10.c101n1cd01
Cross-correlating night10/night10.c102n1cd01
Cross-correlating night10/night10.c105n1cd01
Cross-correlating night10/night10.c106n1cd01
Cross-correlating night10/night10.c109n1cd01
Cross-correlating night10/night10.c110n1cd01
Cross-correlating night10/night10.c113n1cd01
Cross-correlating night10/night10.c114n1cd01
Cross-correlating night10/night10.c117n1cd01
Cross-correlating night10/night10.c118n1cd01
Cross-correlating night10/night10.c121n1cd01
Cross-correlating night10/night10.c122n1cd01
Cross-correlating night10/night10.c125n1cd01
Cross-correlating night10/night10.c126n1cd01
Cross-corr

...

Cross-correlating night5/night5.c226n5c138
Cross-correlating night5/night5.c227n5c138
Cross-correlating night5/night5.c230n5c138
Cross-correlating night5/night5.c231n5c138
Cross-correlating night5/night5.c234n5c138
Cross-correlating night6/night6.c104n5c138
Cross-correlating night6/night6.c105n5c138
Cross-correlating night6/night6.cd01n5c138
Cross-correlating night6/night6.c110n5c138
Cross-correlating night6/night6.c113n5c138
Cross-correlating night6/night6.c114n5c138
Cross-correlating night6/night6.c117n5c138
Cross-correlating night6/night6.c118n5c138
Cross-correlating night6/night6.c121n5c138
Cross-correlating night6/night6.c122n5c138
Cross-correlating night6/night6.c125n5c138
Cross-correlating night6/night6.c126n5c138
Cross-correlating night6/night6.cd02n5c138
Cross-correlating night6/night6.c131n5c138
Cross-correlating night6/night6.c134n5c138
Cross-correlating night6/night6.c135n5c138
Cross-correlating night6/night6.c138n5c138
Cross-correlating night6/night6.c139n5c138
Cross-corre

Cross-correlating night11/night11.c120n5c138
Cross-correlating night11/night11.c121n5c138
Cross-correlating night11/night11.c124n5c138
Cross-correlating night11/night11.c125n5c138
Cross-correlating night11/night11.c128n5c138
Cross-correlating night11/night11.c129n5c138
Cross-correlating night11/night11.c132n5c138
Cross-correlating night11/night11.c133n5c138
Cross-correlating night11/night11.c136n5c138
Cross-correlating night11/night11.c137n5c138
Cross-correlating night11/night11.c204n5c138
Cross-correlating night11/night11.c205n5c138
Cross-correlating night11/night11.c208n5c138
Cross-correlating night11/night11.c209n5c138
Cross-correlating night11/night11.c212n5c138
Cross-correlating night11/night11.c213n5c138
Cross-correlating night11/night11.c216n5c138
Cross-correlating night11/night11.c217n5c138
Cross-correlating night11/night11.c220n5c138
Cross-correlating night11/night11.c221n5c138
Cross-correlating night11/night11.c224n5c138
Cross-correlating night11/night11.c225n5c138
Cross-corr

Cross-correlating night3/night3.c106n5c141
Cross-correlating night3/night3.c107n5c141
Cross-correlating night3/night3.c111n5c141
Cross-correlating night3/night3.c112n5c141
Cross-correlating night3/night3.c115n5c141
Cross-correlating night3/night3.cd02n5c141
Cross-correlating night3/night3.c120n5c141
Cross-correlating night3/night3.c121n5c141
Cross-correlating night3/night3.c124n5c141
Cross-correlating night3/night3.c125n5c141
Cross-correlating night3/night3.c176n5c141
Cross-correlating night3/night3.c177n5c141
Cross-correlating night3/night3.c179n5c141
Cross-correlating night3/night3.c182n5c141
Cross-correlating night3/night3.c183n5c141
Cross-correlating night3/night3.c186n5c141
Cross-correlating night3/night3.c187n5c141
Cross-correlating night3/night3.c190n5c141
Cross-correlating night3/night3.c191n5c141
Cross-correlating night3/night3.c194n5c141
Cross-correlating night3/night3.c197n5c141
Cross-correlating night4/night4.c078n5c141
Cross-correlating night4/night4.c079n5c141
Cross-corre

Killing IRAF task `fxcor'


KeyboardInterrupt: 

In [142]:
full_data = []
for tempnight in obsnights:
    template_list = calibrated_target_template.format(tempnight, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_list)) as templates:
        for template in templates:
            for targnight in obsnights:
                template_cor_file = target_cor_template.format(targnight, obj_types[0], compact_standard(template.upper()))
                
                

nightno = 14
standard_file = calibrated_target_template.format(nightno, obj_types[0])
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_file)) as input_stands:
    standards = input_stands.readlines()
# Each entry in full_measurements will be a column of all measurements of standards.
full_measurements = np.zeros((len(standards), len(standards)))
# True_rv will be an array of the actual RVs of the standards
true_rv = np.zeros(len(standards))
for i, template in enumerate(standards):
    cor_file = target_cor_template.format(nightno, obj_types[0], compact_standard(template).upper())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, cor_file)) as cor_measurements:
        for j, cor in enumerate(cor_measurements):
            if i != j:
                vel_table = Table.read(
                    os.path.join(IMAGE_PATH, CALIB_FOLDER, cor[:-1]+".txt"), format="ascii.commented_header", 
                    fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                    names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 'VOBS', 'VREL', 
                           'VHELIO', 'VERR'])
                rv = vel_table["VHELIO"][-1]
            else:
                rv = int(iraf.hedit(template[:-1], "VHELIO", ".", Stdout=1)[0].split("=")[1])
                true_rv[i] = rv
            full_measurements[i, j] = rv

     

In [144]:
for i in xrange(len(full_measurements)):
    plt.plot(true_rv, full_measurements[i,:], 'k.')
plt.plot([-150, 60], [-150, 60], 'k--')
plt.title("Night {0} Standard Matchup".format(nightno))
plt.xlabel("True RV")
plt.ylabel("Measured RV")

In [139]:
print full_measurements[:, 28]
print true_rv[28]

[-83.2339 -35.3039 -42.5606 -36.6422 -66.707  -40.5319 -64.6345 -36.0654
 -59.8362 -70.9553 -75.1015 -57.4152 -64.8837 -76.6922 -63.1192 -45.7902
 -63.0552 -56.2518 -56.0102 -64.2319 -56.9232 -20.9234 -74.792  -70.91
 -68.605  -32.0534 -55.1465 -55.4843  11.     -64.3785 -57.0747 -39.1069
 -62.2623 -60.038  -61.7525 -41.8154 -55.4474 -37.5308 -64.8943 -32.2136
 -59.7752 -57.7108 -54.684 ]
11.0


In [ ]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspec, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarcs:
        for ref, obj in zip(fullspec, exarcs):
            print obj[:-1]
            iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_arcs)) as calarcs:
    for cstan in calarcs:
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, cstan[:-1]))
        except OSError:
            pass
iraf.dispcor("@"+extracted_arcs, "@"+calibrated_arcs, linearize=True)

In [22]:
# Now apply the wavelength calibration to the images.
# First associate each object spectrum with the arc spectrum.
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspec, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standards)) as stand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarcs, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, argon_specs)) as arspec:
        for ref, obj, arc, ar in zip(fullspec, stand, exarcs, arspec):
            print obj[:-1]
            iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
            iraf.hedit(arc[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
            iraf.hedit(ar[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_arcs)) as calarcs, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_argon_spec)) as calar:
    for cstan in itertools.chain(calstand, calarcs, calar):
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, cstan[:-1]))
        except OSError:
            pass
iraf.dispcor("@"+extracted_standards, "@"+calibrated_standards, linearize=True)
iraf.dispcor("@"+extracted_arcs, "@"+calibrated_arcs, linearize=True)
iraf.dispcor("@"+argon_specs, "@"+calibrated_argon_spec, linearize=True)

night12/night12.076.ms.fits
night12/night12.076.ms.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.076.ms.fits updated
night12/night12.077t076.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.077t076.fits updated
night12/night12.ar076.ms.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.ar076.ms.fits updated
night12/night12.079.ms.fits
night12/night12.079.ms.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.079.ms.fits updated
night12/night12.078t079.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.078t079.fits updated
night12/night12.ar079.ms.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.ar079.ms.fits updated
night12/night12.080.ms.fits
night12/night12.080.ms.fits,REFSPEC1: night12/night12.full

night12/night12.112.ms.fits
night12/night12.112.ms.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.112.ms.fits updated
night12/night12.113t112.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.113t112.fits updated
night12/night12.ar112.ms.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.ar112.ms.fits updated
night12/night12.115.ms.fits
night12/night12.115.ms.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.115.ms.fits updated
night12/night12.114t115.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.114t115.fits updated
night12/night12.ar115.ms.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.ar115.ms.fits updated
night12/night12.116.ms.fits
night12/night12.116.ms.fits,REFSPEC1: night12/night12.full

night12/night12.210.ms.fits
night12/night12.210.ms.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.210.ms.fits updated
night12/night12.211t210.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.211t210.fits updated
night12/night12.ar210.ms.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.ar210.ms.fits updated
night12/night12.213.ms.fits
night12/night12.213.ms.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.213.ms.fits updated
night12/night12.212t213.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.212t213.fits updated
night12/night12.ar213.ms.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.ar213.ms.fits updated
night12/night12.214.ms.fits
night12/night12.214.ms.fits,REFSPEC1: night12/night12.full

night12/night12.099.ms.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.c099.ms.fit: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.100.ms.fits: REFSPEC1 = 'night12/night12.full100.ms.fits 1.'
night12/night12.c100.ms.fit: ap = 1, w1 = 4346.648, w2 = 6062.877, dw =  1.01014, nw = 1700
night12/night12.103.ms.fits: REFSPEC1 = 'night12/night12.full103.ms.fits 1.'
night12/night12.c103.ms.fit: ap = 1, w1 = 4348.703, w2 = 6062.816, dw = 1.008895, nw = 1700
night12/night12.104.ms.fits: REFSPEC1 = 'night12/night12.full104.ms.fits 1.'
night12/night12.c104.ms.fit: ap = 1, w1 = 4347.442, w2 = 6062.802, dw =  1.00963, nw = 1700
night12/night12.107.ms.fits: REFSPEC1 = 'night12/night12.full107.ms.fits 1.'
night12/night12.c107.ms.fit: ap = 1, w1 = 4346.895, w2 = 6062.852, dw = 1.009981, nw = 1700
night12/night12.108.ms.fits: REFSPEC1 = 'night12/night12.full108.ms.fits 1.'
night12/night12.c108.ms.fit: ap = 1, w1 = 4347.546, w2 =  6062.85, dw = 1.0095

night12/night12.c094t095.ms.fits: ap = 1, w1 = 4347.328, w2 = 6062.814, dw = 1.009704, nw = 1700
night12/night12.097t096.fits: REFSPEC1 = 'night12/night12.full096.ms.fits 1.'
night12/night12.c097t096.ms.fits: ap = 1, w1 =   4347.3, w2 = 6062.816, dw = 1.009721, nw = 1700
night12/night12.098t099.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.c098t099.ms.fits: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.101t100.fits: REFSPEC1 = 'night12/night12.full100.ms.fits 1.'
night12/night12.c101t100.ms.fits: ap = 1, w1 = 4346.648, w2 = 6062.877, dw =  1.01014, nw = 1700
night12/night12.102t103.fits: REFSPEC1 = 'night12/night12.full103.ms.fits 1.'
night12/night12.c102t103.ms.fits: ap = 1, w1 = 4348.703, w2 = 6062.816, dw = 1.008895, nw = 1700
night12/night12.105t104.fits: REFSPEC1 = 'night12/night12.full104.ms.fits 1.'
night12/night12.c105t104.ms.fits: ap = 1, w1 = 4347.442, w2 = 6062.802, dw =  1.00963, nw = 1700
night12/night12.106t107.fits

night12/night12.car088.fits: ap = 1, w1 = 4346.902, w2 = 6062.873, dw = 1.009989, nw = 1700
night12/night12.ar091.ms.fits: REFSPEC1 = 'night12/night12.full091.ms.fits 1.'
night12/night12.car091.fits: ap = 1, w1 = 4346.863, w2 = 6062.853, dw =     1.01, nw = 1700
night12/night12.ar092.ms.fits: REFSPEC1 = 'night12/night12.full092.ms.fits 1.'
night12/night12.car092.fits: ap = 1, w1 = 4346.607, w2 = 6062.877, dw = 1.010165, nw = 1700
night12/night12.ar095.ms.fits: REFSPEC1 = 'night12/night12.full095.ms.fits 1.'
night12/night12.car095.fits: ap = 1, w1 = 4347.328, w2 = 6062.814, dw = 1.009704, nw = 1700
night12/night12.ar096.ms.fits: REFSPEC1 = 'night12/night12.full096.ms.fits 1.'
night12/night12.car096.fits: ap = 1, w1 =   4347.3, w2 = 6062.816, dw = 1.009721, nw = 1700
night12/night12.ar099.ms.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.car099.fits: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.ar100.ms.fits: REFSPEC1 = 'night12/ni

In [12]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarcs:
        for stand, arcname in zip(calstand, exarcs):
            crosscor = os.path.splitext(arcname)[0] + ".txt"
            shift_table = Table.read(crosscor, format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                     header_start=13, guess=False, 
                                     names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                            'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
            iraf.hedit(stand[:-1], "CRPIX1", "(1-{0:g})".format(shift_table["SHIFT"][-1]), verify=False)

IOError: [Errno 2] No such file or directory: 'night12/night12.077t076.ms.txt'

In [40]:
standard_photometry = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "Standard_Photometry.txt"), 
                            format="ascii.commented_header", header_start=0, data_start=4, data_end=-1, delimiter="|", guess=False)
standard_info = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "standard_csv.csv"))
standards_JK = standard_photometry["Mag J"] - standard_photometry["Mag K"]
jktable = Table([standard_photometry["typed ident"], standards_JK], names=("Star", "J-K"))
standard_info = join(jktable, standard_info, keys=["Star"])
standard_info.sort("J-K")

In [41]:
lookup = collections.defaultdict(list)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand:
    for stand in calstand:
        hdulist = fits.open(os.path.join(IMAGE_PATH, CALIB_FOLDER, stand[:-1]))
        imgname = hdulist[0].header["OBJECT"]
        hdulist.close()
        lookup[imgname] = stand[:-1]
lookup_table = Table(rows=lookup.items(), names=("Star", "SpecPath"))
standard_info = join(standard_info, lookup_table, keys=["Star"], join_type="right")

In [86]:
for standname, rvel, standfile in zip(standard_info["Star"], standard_info["RV"], standard_info["SpecPath"]):
    iraf.hedit(standfile, "VHELIO", rvel, add=True, verify=False)

night12/night12.c226.ms.fit,VHELIO: -86 -> -86
night12/night12.c226.ms.fit updated
add night12/night12.c127.ms.fit,VHELIO = -16
night12/night12.c127.ms.fit updated
add night12/night12.c131.ms.fit,VHELIO = -29
night12/night12.c131.ms.fit updated
add night12/night12.c083.ms.fit,VHELIO = -19
night12/night12.c083.ms.fit updated
add night12/night12.c087.ms.fit,VHELIO = 5
night12/night12.c087.ms.fit updated
night12/night12.c213.ms.fit,VHELIO: -47 -> -47
night12/night12.c213.ms.fit updated
night12/night12.c234.ms.fit,VHELIO: -60 -> -60
night12/night12.c234.ms.fit updated
add night12/night12.c107.ms.fit,VHELIO = -12
night12/night12.c107.ms.fit updated
add night12/night12.c104.ms.fit,VHELIO = -2
night12/night12.c104.ms.fit updated
add night12/night12.c140.ms.fit,VHELIO = 36
night12/night12.c140.ms.fit updated
add night12/night12.c139.ms.fit,VHELIO = 15
night12/night12.c139.ms.fit updated
add night12/night12.c095.ms.fit,VHELIO = -39
night12/night12.c095.ms.fit updated
night12/night12.c222.ms.fit

In [16]:
template_standard = standard_info["Star"][len(standard_info)/2]
template_file = standard_info["SpecPath"][len(standard_info)/2]
# Get the index of the number in the name, which is assumed to be in the form of "night12/night12.c076.ms.fits"
template_num_index = template_file.index(".c")+2
template_num = template_file[template_num_index:template_num_index+3]

In [26]:
iraf.keywpar.ut = "TIME-OBS"
iraf.keywpar.epoch = "EQUINOX"

iraf.fxcor.pixcorr = False
iraf.function = "gaussian"

iraf.continpars.c_inter = True
iraf.continpars.order = 10
iraf.continpars.low_rej = 2
iraf.continpars.high_rej = 5
iraf.continpars.nitera = 10
iraf.continpars.grow = 1

iraf.filtpars.cutoff = 250
iraf.filtpars.cuton = 25

for objrow in standard_info:
    object_file = objrow["SpecPath"]
    object_num_index = object_file.index(".c")+2
    object_num = object_file[object_num_index:object_num_index+3]
    
    output_root = os.path.join("night{0:d}", "night{0}.{1}cc{2}").format(Calib_Night, object_num, template_num)
    print(output_root)
    iraf.fxcor(object_file, template_file, output=output_root)
    

night12/night12.226cc096
Cross-Correlating night12/night12.c226.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.9757  FWHM=304.43  Vr=-119.313  Vo=-101.115  Vh=-85.154 +/- 7.615
HJD=7915.9757  FWHM=330.17  Vr=-119.356  Vo=-101.159  Vh=-85.197 +/- 8.291
Writing current results to `night12/night12.226cc096.txt'....Done.
night12/night12.127cc096
Cross-Correlating night12/night12.c127.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.7002  FWHM=306.73  Vr=-9.204  Vo=9.001  Vh=-7.947 +/- 8.071
HJD=7915.7002  FWHM=326.53  Vr=-9.234  Vo=8.970  Vh=-7.977 +/- 8.618
Writing current results to `night12/night12.127cc096.txt'....Done.
night12/night12.131cc096
Cross-Correlating night12/night12.c131.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.7036  FWHM=306.50  Vr=-41.243  Vo=-23.041  Vh=-33.606 +/- 8.105
HJD=7915.7036  FWHM=327.67  Vr=-41.270  Vo=-23.068  Vh=-33.633 +/- 8.693
Writing current results to `night12/night12.131cc096.txt'....Done.
night12/night12.083cc096
Cross-C

HJD=7915.6642  FWHM=336.70  Vr=20.185  Vo=38.391  Vh=12.952 +/- 6.557
Writing current results to `night12/night12.099cc096.txt'....Done.


In [42]:
standard_info.sort("J-K")
cor_vels = np.zeros(len(standard_info))
for i, specfile in enumerate(standard_info["SpecPath"]):  
    # Now get the pixel shift.
    num_index = specfile.index(".c")+2
    specnum = specfile[num_index:num_index+3]
    shiftfile = os.path.join("night{0}", "night{0}.{1}cc{2}.txt").format(Calib_Night, specnum, template_num)
    shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile), format="ascii.commented_header", 
                             fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                    'VOBS', 'VREL', 'VHELIO', 'VERR'])
    cor_vels[i] = shift_table["VHELIO"][-1]

In [43]:
veldiffs = cor_vels - standard_info["RV"]
plt.plot(standard_info["J-K"], veldiffs, 'ko')
plt.plot(standard_info["J-K"][len(standard_info)/2], veldiffs[len(veldiffs)/2], 'r*', ms=10)
plt.xlabel("J-K")
plt.ylabel("Relative velocity offset")

In [52]:
standard_info["Star"][len(standard_info)/2]

'HD110044'

In [24]:
standard_info["RV"][len(standard_info)/2]

Star,J-K,Spectype,RV,V,SpecPath
str9,float64,str8,int64,float64,str27
HD110044,0.455,K1V,-7,9.0,night12/night12.c096.ms.fit


In [48]:
plt.plot(cor_vels, standard_info["RV"], 'ko')
plt.xlabel("Cross-correlated RV (km/s)")
plt.ylabel("Literature RV (km/s)")

# Cross-correlation

In [7]:
def compact_standard(filename):
    '''Compactify a filename.'''
    compact = os.path.splitext(os.path.splitext(os.path.basename(filename))[0])[0].replace(".", "").replace("night","n")
    return compact
# fxcor_velocity_base.format(12, compact("night12/night12.c122.ms.fits").upper()) -> "Night12_FXCor_N12C122.txt"
fxcor_velocity_base = "Night{0:d}_FXCor_{1}.txt"

In [35]:
template_standard = "night12/night12.c096.ms.fit"
iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"
for n in obsnights:
    kicfiles = calibrated_target_template.format(n, obj_types[1])
    velocity_files = fxcor_velocity_base.format(n, compact_standard(template_standard).upper())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kicfiles), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, velocity_files), 'w') as newfile:
            for oldname in oldfile:
                compact = compact_standard(template_standard)
                kicnum = oldname[-12:-9]
                newname = os.path.splitext(os.path.splitext(oldname)[0])[0].replace("c"+kicnum, kicnum+"cc"+compact)
                newfile.write(newname+"\n")
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kicfiles)) as kic_targets, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, velocity_files)) as output_files:
            for kic_target, output in zip(kic_targets, output_files):
                print kic_target, output
                iraf.fxcor(kic_target[:-1], template_standard, out=output[:-1], intera="no")

night1/night1.c081.ms.fits
night1/night1.081ccn12c096

night1/night1.c082.ms.fits
night1/night1.082ccn12c096

night1/night1.c083.ms.fits
night1/night1.083ccn12c096

night1/night1.c084.ms.fits
night1/night1.084ccn12c096

night1/night1.c085.ms.fits
night1/night1.085ccn12c096

night1/night1.c086.ms.fits
night1/night1.086ccn12c096

night1/night1.c087.ms.fits
night1/night1.087ccn12c096

night1/night1.c089.ms.fits
night1/night1.089ccn12c096

night1/night1.c090.ms.fits
night1/night1.090ccn12c096

night1/night1.c091.ms.fits
night1/night1.091ccn12c096

night1/night1.c092.ms.fits
night1/night1.092ccn12c096

night1/night1.c093.ms.fits
night1/night1.093ccn12c096

night1/night1.c094.ms.fits
night1/night1.094ccn12c096

night1/night1.c095.ms.fits
night1/night1.095ccn12c096

night1/night1.c097.ms.fits
night1/night1.097ccn12c096

night1/night1.c098.ms.fits
night1/night1.098ccn12c096

night1/night1.c099.ms.fits
night1/night1.099ccn12c096

night1/night1.c100.ms.fits
night1/night1.100ccn12c096

night1/nig

night6/night6.c197.ms.fits
night6/night6.197ccn12c096

night6/night6.c198.ms.fits
night6/night6.198ccn12c096

night6/night6.c199.ms.fits
night6/night6.199ccn12c096

night6/night6.c201.ms.fits
night6/night6.201ccn12c096

night6/night6.c202.ms.fits
night6/night6.202ccn12c096

night6/night6.c203.ms.fits
night6/night6.203ccn12c096

night6/night6.c206.ms.fits
night6/night6.206ccn12c096

night6/night6.c207.ms.fits
night6/night6.207ccn12c096

night6/night6.c208.ms.fits
night6/night6.208ccn12c096

night6/night6.c209.ms.fits
night6/night6.209ccn12c096

night6/night6.c210.ms.fits
night6/night6.210ccn12c096

night6/night6.c212.ms.fits
night6/night6.212ccn12c096

night6/night6.c213.ms.fits
night6/night6.213ccn12c096

night6/night6.c214.ms.fits
night6/night6.214ccn12c096

night6/night6.c215.ms.fits
night6/night6.215ccn12c096

night6/night6.c216.ms.fits
night6/night6.216ccn12c096

night6/night6.c218.ms.fits
night6/night6.218ccn12c096

night6/night6.c219.ms.fits
night6/night6.219ccn12c096

night6/nig

night10/night10.c157.ms.fits
night10/night10.157ccn12c096

night10/night10.c158.ms.fits
night10/night10.158ccn12c096

night10/night10.c159.ms.fits
night10/night10.159ccn12c096

night10/night10.c161.ms.fits
night10/night10.161ccn12c096

night10/night10.c162.ms.fits
night10/night10.162ccn12c096

night10/night10.c163.ms.fits
night10/night10.163ccn12c096

night10/night10.c164.ms.fits
night10/night10.164ccn12c096

night10/night10.c165.ms.fits
night10/night10.165ccn12c096

night10/night10.c166.ms.fits
night10/night10.166ccn12c096

night10/night10.c168.ms.fits
night10/night10.168ccn12c096

night10/night10.c169.ms.fits
night10/night10.169ccn12c096

night10/night10.c170.ms.fits
night10/night10.170ccn12c096

night10/night10.c171.ms.fits
night10/night10.171ccn12c096

night10/night10.c172.ms.fits
night10/night10.172ccn12c096

night10/night10.c174.ms.fits
night10/night10.174ccn12c096

night10/night10.c175.ms.fits
night10/night10.175ccn12c096

night10/night10.c176.ms.fits
night10/night10.176ccn12c09

night13/night13.c145.ms.fits
night13/night13.145ccn12c096

night13/night13.c146.ms.fits
night13/night13.146ccn12c096

night13/night13.c147.ms.fits
night13/night13.147ccn12c096

night13/night13.c148.ms.fits
night13/night13.148ccn12c096

night13/night13.c150.ms.fits
night13/night13.150ccn12c096

night13/night13.c151.ms.fits
night13/night13.151ccn12c096

night13/night13.c152.ms.fits
night13/night13.152ccn12c096

night13/night13.c153.ms.fits
night13/night13.153ccn12c096

night13/night13.c154.ms.fits
night13/night13.154ccn12c096

night13/night13.c155.ms.fits
night13/night13.155ccn12c096

night13/night13.c157.ms.fits
night13/night13.157ccn12c096

night13/night13.c158.ms.fits
night13/night13.158ccn12c096

night13/night13.c159.ms.fits
night13/night13.159ccn12c096

night13/night13.c160.ms.fits
night13/night13.160ccn12c096

night13/night13.c161.ms.fits
night13/night13.161ccn12c096

night13/night13.c162.ms.fits
night13/night13.162ccn12c096

night13/night13.c164.ms.fits
night13/night13.164ccn12c09

In [66]:
RVs = collections.defaultdict(list)
for n in obsnights:
    RV_night = collections.defaultdict(list)
    kic_objects = calibrated_target_template.format(n, obj_types[1])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_objects)) as kic_targets:
        for kicfile in kic_targets:
            compact = compact_standard(template_standard)
            kicnum = kicfile[-12:-9]
            velfile = os.path.splitext(os.path.splitext(kicfile)[0])[0].replace("c"+kicnum, kicnum+"cc"+compact)+".txt"
            vel_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, velfile), format="ascii.commented_header", 
                                   fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                                   names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                          'VOBS', 'VREL', 'VHELIO', 'VERR'])
            if abs(vel_table["VHELIO"][-1]) > 500:
                print "Strange velocity for {0}:{1}".format(vel_table["IMAGE"][-1], vel_table["VHELIO"][-1])
            else:
                RV_night[vel_table["OBJECT"][-1]].append(vel_table["VHELIO"][-1])
                print "{0}: {1}".format(vel_table["OBJECT"][-1], vel_table["IMAGE"][-1])
        for k,v in RV_night.iteritems():
            mean = np.mean(v)
            RVs[k].append(mean)

KIC6844101: night1/night1.c081.ms.fits
KIC6844101: night1/night1.c082.ms.fits
KIC1570924: night1/night1.c083.ms.fits
KIC9653110: night1/night1.c084.ms.fits
Strange velocity for night1/night1.c085.ms.fits:27615.646
KIC7421325: night1/night1.c086.ms.fits
KIC8651471: night1/night1.c087.ms.fits
KIC5553362: night1/night1.c089.ms.fits
KIC5213142: night1/night1.c090.ms.fits
KIC3539632: night1/night1.c091.ms.fits
Strange velocity for night1/night1.c092.ms.fits:-7471.932
KIC11819949: night1/night1.c093.ms.fits
KIC12736892: night1/night1.c094.ms.fits
KIC8442720: night1/night1.c095.ms.fits
Strange velocity for night1/night1.c097.ms.fits:22666.102
Strange velocity for night1/night1.c098.ms.fits:-24050.21
Strange velocity for night1/night1.c099.ms.fits:32247.787
Strange velocity for night1/night1.c100.ms.fits:23341.089
KIC3540728: night1/night1.c101.ms.fits
KIC5609753: night1/night1.c102.ms.fits
KIC6780052: night1/night1.c103.ms.fits
KIC4036736: night1/night1.c106.ms.fits
KIC4480434: night1/night1.

KIC5213142: night9/night9.cd11.ms.fits
Strange velocity for night9/night9.cd12.ms.fits:-19196.3
KIC9151271: night9/night9.cd13.ms.fits
KIC8651471: night9/night9.cd14.ms.fits
KIC11819949: night9/night9.cd15.ms.fits
KIC9653110: night9/night9.cd16.ms.fits
KIC8442720: night9/night9.cd17.ms.fits
KIC4480434: night9/night9.cd18.ms.fits
KIC6780052: night9/night9.cd19.ms.fits
KIC9964938: night9/night9.cd20.ms.fits
KIC6425783: night9/night9.cd21.ms.fits
KIC4249702: night9/night9.cd22.ms.fits
KIC5609753: night9/night9.cd23.ms.fits
KIC6844101: night9/night9.cd24.ms.fits
KIC7421325: night9/night9.cd25.ms.fits
KIC3219623: night9/night9.cd26.ms.fits
KIC10802309: night9/night9.cd27.ms.fits
KIC9655045: night9/night9.cd28.ms.fits
KIC3539632: night9/night9.cd29.ms.fits
KIC7919763: night9/night9.cd30.ms.fits
KIC4036736: night9/night9.cd31.ms.fits
KIC11073910: night9/night9.cd32.ms.fits
KIC9151271: night9/night9.cd33.ms.fits
KIC5609753: night9/night9.cd34.ms.fits
KIC9964938: night10/night10.cd01.ms.fits
KI

KIC9964938: night13/night13.c197.ms.fits
KIC3539632: night14/night14.c134.ms.fits
KIC5609753: night14/night14.c135.ms.fits
Strange velocity for night14/night14.c136.ms.fits:31522.944
KIC9151271: night14/night14.c137.ms.fits
KIC5213142: night14/night14.c138.ms.fits
KIC4036736: night14/night14.c140.ms.fits
KIC5609753: night14/night14.c141.ms.fits
KIC3219623: night14/night14.c142.ms.fits
KIC10153521: night14/night14.c143.ms.fits
KIC8651471: night14/night14.c144.ms.fits
KIC4454890: night14/night14.c146.ms.fits
KIC8442720: night14/night14.c147.ms.fits
KIC11819949: night14/night14.c148.ms.fits
KIC9151271: night14/night14.c149.ms.fits
KIC6780052: night14/night14.c150.ms.fits
KIC9653110: night14/night14.c151.ms.fits
KIC5609753: night14/night14.c153.ms.fits
KIC6844101: night14/night14.c154.ms.fits
KIC7919763: night14/night14.c155.ms.fits
KIC9710336: night14/night14.c156.ms.fits
KIC1570924: night14/night14.c157.ms.fits
KIC6425783: night14/night14.c159.ms.fits
KIC4249702: night14/night14.c160.ms.

In [43]:
print RVs

defaultdict(<type 'list'>, {'KIC9964938': [-2.5164999999999997, -7.0715500000000002, -7.6976000000000004, -5.6304999999999996, 8.0273000000000003, 3.2827000000000002, -7.5427999999999997, -6.9744499999999992, -15.0952, -8.9543999999999997, -8.5599999999999987, -6.7424999999999997], 'KIC6844101': [-29.983800000000002, -29.0213, -26.5608, -15.273300000000001, -20.3413, -13.617599999999999, -15.4138, -26.4041, -25.684799999999999, -26.949300000000001, -22.317700000000002, -26.69755, -30.956600000000002], 'KIC9710336': [-43.990900000000003, -3.2900499999999999, -23.2502, -27.072299999999998, -34.471499999999999, -23.785499999999999, -5.9043999999999999, -27.7745, -31.365300000000001, -31.10905, 34.432600000000001, -28.010149999999996, -39.762600000000006], 'KIC9653110': [-51.360500000000002, -18.1813, -29.255199999999999, -20.181249999999999, -38.868299999999998, -38.732199999999999, -30.851500000000001, -43.973799999999997, -52.823499999999996, -34.471450000000004, -37.203850000000003, -5

In [67]:
kicnum = []
std = []
means = []
periodlist = []
periods = {"KIC1570924": 3.2, "KIC3540728": 2.1, "KIC5553362": 4.4, "KIC7294867": 12.3, "KIC8442720": 3.5, 
           "KIC9964938": 2.8, "KIC10293980": 1.0, "KIC11073910": 2.0, "KIC11819949": 2.3, "KIC12736892": 2.6, 
           "KIC8651471": 3.4, "KIC4249702": 4.7, "KIC10802309": 1.9, "KIC6780052": 3.1, "KIC3539632": 3.1, 
           "KIC4480434": 4.5, "KIC7919763": 3.7, "KIC4454890": 2.2, "KIC3248885": 4.8, "KIC5213142": 2.5, 
           "KIC9710336": 4.4, "KIC10153521": 1.7, "KIC4036736": 2.8, "KIC6844101": 2.5, "KIC11080481": 1.0, 
           "KIC3219623": 1.6, "KIC6425783": 3.9, "KIC9655045": 2.9, "KIC9653110": 3.1, "KIC7421325": 4.7, 
           "KIC9151271": 4.6, "KIC5609753": 3.2}
for k,v in RVs.iteritems():
    kicnum.append(int(k[3:]))
    std.append(np.std(v))
    means.append(np.mean(v))
    periodlist.append(periods[k])

In [70]:
plt.plot(periodlist, std, 'ko')
plt.xlabel("Rotation Period (day)")
plt.ylabel("RV Standard Deviation (km/s)")
plt.title("Preliminary RV Variability for sample")

In [62]:
RVs["KIC3540729"]

[-22.866]